# Validate T-Web Eigenvalue Outputs (CubicBox)

This notebook compares two AbacusSummit T-Web output sets:

- `/pscratch/sd/d/dkololgi/AbscusSummit_densities/tweb_rank_outputs`
- `/pscratch/sd/d/dkololgi/AbscusSummit_densities/tweb_rank_outputs_old`

It runs quantitative checks and produces plots for:

1. Slab metadata consistency (`ngrid`, `boxsize`, `threshold`, `Rsmooth`, slab ranges)
2. Eigenvalue distribution sanity (`lambda1/2/3` histograms and CDFs)
3. Eigenvalue ordering validity (`lambda1 <= lambda2 <= lambda3`)
4. CWEB class fractions and differences
5. Local grid continuity diagnostics (neighbor vs random)
6. Cross-version agreement (MAE/RMSE/correlation per eigenvalue, CWEB agreement)

All plots are saved to a timestamped output directory.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from datetime import datetime
import json
import glob

import numpy as np
import matplotlib.pyplot as plt

# Optional convenience for tabular summaries.
try:
    import pandas as pd
except Exception:
    pd = None

plt.style.use("dark_background")

# NEW_DIR = Path('/pscratch/sd/d/dkololgi/AbacusSummit_densities/tweb_rank_outputs_fullgrid_v2/backend_optimized_ngrid_2048_rsmooth_4')
NEW_DIR = Path('/pscratch/sd/d/dkololgi/AbacusSummit_densities/tweb_rank_outputs_fullgrid_v3/dens_AbacusSummit_base_c000_ph000_z0.200_ngrid2048_box2000_thr0p2/backend_optimized_ngrid_2048_rsmooth_8')
# NEW_DIR = Path('/pscratch/sd/d/dkololgi/AbacusSummit_densities/tweb_rank_outputs_fullgrid_v3/dens_AbacusSummit_base_c000_ph000_z0.200_ngrid2048_box2000_thr0p2/backend_optimized_ngrid_2048_rsmooth_12')
# Path('/pscratch/sd/d/dkololgi/AbscusSummit_densities/tweb_rank_outputs')# _ng512_rs8')
# Path('/pscratch/sd/d/dkololgi/AbscusSummit_densities/tweb_rank_outputs')

# Fast single-run physical sanity mode (recommended default).
PRIMARY_DIR = NEW_DIR
COMPARE_OLD = False
FAST_QC = True
RUN_HEAVY_CROSS_VERSION = False

# Tunables for faster turnaround.
FAST_MAX_STREAM_CELLS = 5_000_000
FAST_SAMPLE_PLOT = 120_000
FAST_CONTINUITY_SAMPLE = 20_000

OUT_DIR = Path('/pscratch/sd/d/dkololgi/abacus/alignment_diagnostics') / f"tweb_compare_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"PRIMARY_DIR: {PRIMARY_DIR}")
print(f"COMPARE_OLD: {COMPARE_OLD}")
print(f"FAST_QC: {FAST_QC}")
print(f"RUN_HEAVY_CROSS_VERSION: {RUN_HEAVY_CROSS_VERSION}")
print(f"OUT_DIR: {OUT_DIR}")

In [ ]:
from astropy.table import Table
from astropy.io import fits

PATH2MOCKTWEB2= '/pscratch/sd/d/dkololgi/abacus/mocks_with_eigs/cutsky_BGS_z0.200_AbacusSummit_base_c000_ph000_with_tweb_eigs_ng2048_rs4_v2.fits'
data = Table(fits.open(PATH2MOCKTWEB2))
data[(data['IN_Y1']==1) & (data['IN_Y5'])==1]

In [ ]:
@dataclass(frozen=True)
class SlabMeta:
    rank: int
    path: Path
    x_start: int
    x_end: int
    ngrid: int
    boxsize: float
    threshold: float
    rsmooth: float


def discover_slabs(tweb_dir: Path) -> list[SlabMeta]:
    files = sorted(glob.glob(str(tweb_dir / 'abacus_cactus_tweb_rank*.npz')))
    slabs: list[SlabMeta] = []
    for p in files:
        rank = int(Path(p).stem.split('rank')[-1])
        with np.load(p) as d:
            slabs.append(
                SlabMeta(
                    rank=rank,
                    path=Path(p),
                    x_start=int(d['x_start']),
                    x_end=int(d['x_end']),
                    ngrid=int(d['ngrid']),
                    boxsize=float(d['boxsize']),
                    threshold=float(d['threshold']),
                    rsmooth=float(d['Rsmooth']),
                )
            )
    slabs.sort(key=lambda s: s.x_start)
    return slabs


def validate_slab_layout(slabs: list[SlabMeta]) -> dict:
    if not slabs:
        return {'ok': False, 'reason': 'no slabs'}

    ngrid = slabs[0].ngrid
    expected = 0
    gaps = []
    overlaps = []
    for s in slabs:
        if s.x_start > expected:
            gaps.append((expected, s.x_start))
        if s.x_start < expected:
            overlaps.append((s.x_start, expected))
        expected = s.x_end

    complete = (expected == ngrid)
    return {
        'ok': (len(gaps) == 0 and len(overlaps) == 0 and complete),
        'n_slabs': len(slabs),
        'ngrid': ngrid,
        'boxsize': slabs[0].boxsize,
        'threshold': slabs[0].threshold,
        'rsmooth': slabs[0].rsmooth,
        'coverage_end': expected,
        'complete': complete,
        'gaps': gaps,
        'overlaps': overlaps,
    }


def stream_stats(slabs: list[SlabMeta], max_cells: int | None = None) -> dict:
    """Compute streaming moments and class counts without loading all slabs at once."""
    sums = np.zeros(3, dtype=np.float64)
    sums2 = np.zeros(3, dtype=np.float64)
    mins = np.array([np.inf, np.inf, np.inf], dtype=np.float64)
    maxs = np.array([-np.inf, -np.inf, -np.inf], dtype=np.float64)
    class_counts = np.zeros(4, dtype=np.int64)
    order_viol_12 = 0
    order_viol_23 = 0
    total = 0

    for s in slabs:
        with np.load(s.path) as d:
            eig = np.asarray(d['eig_vals'], dtype=np.float32)  # [3, nx, ny, nz]
            cweb = np.asarray(d['cweb'])
        vals = np.moveaxis(eig, 0, -1).reshape(-1, 3)  # [N, 3]
        cls = cweb.reshape(-1)

        if max_cells is not None and total + vals.shape[0] > max_cells:
            keep = max_cells - total
            if keep <= 0:
                break
            vals = vals[:keep]
            cls = cls[:keep]

        sums += vals.sum(axis=0)
        sums2 += np.square(vals).sum(axis=0)
        mins = np.minimum(mins, vals.min(axis=0))
        maxs = np.maximum(maxs, vals.max(axis=0))
        total += vals.shape[0]

        for k in range(4):
            class_counts[k] += int(np.sum(cls == k))

        order_viol_12 += int(np.sum(vals[:, 0] > vals[:, 1]))
        order_viol_23 += int(np.sum(vals[:, 1] > vals[:, 2]))

        if max_cells is not None and total >= max_cells:
            break

    mean = sums / max(total, 1)
    var = sums2 / max(total, 1) - np.square(mean)
    std = np.sqrt(np.maximum(var, 0.0))

    return {
        'n_cells': int(total),
        'mean': mean.tolist(),
        'std': std.tolist(),
        'min': mins.tolist(),
        'max': maxs.tolist(),
        'class_counts': class_counts.tolist(),
        'order_viol_l1_gt_l2': int(order_viol_12),
        'order_viol_l2_gt_l3': int(order_viol_23),
        'order_ok_fraction': float(1.0 - (order_viol_12 + order_viol_23) / max(1, 2 * total)),
    }


def continuity_ratio_on_sample(slabs: list[SlabMeta], sample_cells: int = 50000, k: int = 6, seed: int = 42) -> dict:
    """Local continuity on random sample of cells inside each version."""
    rng = np.random.default_rng(seed)
    xyz_list = []
    eig_list = []

    # Sample proportionally by slab volume.
    vols = np.array([(s.x_end - s.x_start) * s.ngrid * s.ngrid for s in slabs], dtype=np.float64)
    probs = vols / vols.sum()

    picks = rng.multinomial(sample_cells, probs)
    for s, n_pick in zip(slabs, picks):
        if n_pick == 0:
            continue
        with np.load(s.path) as d:
            eig = np.asarray(d['eig_vals'], dtype=np.float32)
        nx = s.x_end - s.x_start
        ix = rng.integers(0, nx, size=n_pick)
        iy = rng.integers(0, s.ngrid, size=n_pick)
        iz = rng.integers(0, s.ngrid, size=n_pick)
        lam = np.stack([eig[0, ix, iy, iz], eig[1, ix, iy, iz], eig[2, ix, iy, iz]], axis=1)
        xyz = np.stack([ix + s.x_start, iy, iz], axis=1).astype(np.float32)
        xyz_list.append(xyz)
        eig_list.append(lam)

    xyz = np.concatenate(xyz_list, axis=0)
    lam = np.concatenate(eig_list, axis=0)

    try:
        from sklearn.neighbors import NearestNeighbors  # type: ignore

        nn = NearestNeighbors(n_neighbors=min(k + 1, xyz.shape[0]))
        nn.fit(xyz)
        idx = nn.kneighbors(return_distance=False)[:, 1:]
    except Exception:
        # Fallback brute force if sklearn unavailable.
        n = min(3000, xyz.shape[0])
        xyz = xyz[:n]
        lam = lam[:n]
        d2 = np.sum((xyz[:, None, :] - xyz[None, :, :]) ** 2, axis=2)
        np.fill_diagonal(d2, np.inf)
        idx = np.argpartition(d2, kth=min(k, n - 1), axis=1)[:, : min(k, n - 1)]

    neigh = np.mean(np.abs(lam[:, None, :] - lam[idx, :]), axis=(1, 2)).mean()
    j = rng.integers(0, lam.shape[0], size=lam.shape[0])
    rand = np.mean(np.abs(lam - lam[j]), axis=1).mean()
    return {
        'neighbor_absdiff_mean': float(neigh),
        'random_absdiff_mean': float(rand),
        'continuity_ratio_neighbor_over_random': float(neigh / max(rand, 1e-12)),
        'n_sampled': int(lam.shape[0]),
    }


def compare_versions(new_slabs: list[SlabMeta], old_slabs: list[SlabMeta]) -> dict:
    """Cell-wise comparison where slab rank and shape align."""
    old_by_rank = {s.rank: s for s in old_slabs}
    mae_sum = np.zeros(3, dtype=np.float64)
    mse_sum = np.zeros(3, dtype=np.float64)
    n = 0
    cweb_eq = 0

    for ns in new_slabs:
        os = old_by_rank.get(ns.rank)
        if os is None:
            continue
        with np.load(ns.path) as dn, np.load(os.path) as do:
            en = np.asarray(dn['eig_vals'], dtype=np.float32)
            eo = np.asarray(do['eig_vals'], dtype=np.float32)
            cn = np.asarray(dn['cweb'])
            co = np.asarray(do['cweb'])

        if en.shape != eo.shape or cn.shape != co.shape:
            continue

        diff = (en - eo).reshape(3, -1).T
        mae_sum += np.mean(np.abs(diff), axis=0) * diff.shape[0]
        mse_sum += np.mean(np.square(diff), axis=0) * diff.shape[0]
        cweb_eq += int(np.sum(cn == co))
        n += diff.shape[0]

    if n == 0:
        return {'n_cells_compared': 0}

    mae = mae_sum / n
    rmse = np.sqrt(mse_sum / n)
    return {
        'n_cells_compared': int(n),
        'mae_lambda': mae.tolist(),
        'rmse_lambda': rmse.tolist(),
        'mae_lambda_mean': float(np.mean(mae)),
        'rmse_lambda_mean': float(np.mean(rmse)),
        'cweb_match_fraction': float(cweb_eq / n),
    }

In [ ]:
primary_slabs = discover_slabs(PRIMARY_DIR)
other_slabs = discover_slabs(OLD_DIR) if COMPARE_OLD else []

print(f"Primary slabs: {len(primary_slabs)} from {PRIMARY_DIR}")
if COMPARE_OLD:
    print(f"Other slabs: {len(other_slabs)} from {OLD_DIR}")

stream_cap = FAST_MAX_STREAM_CELLS if FAST_QC else None
sample_cont = FAST_CONTINUITY_SAMPLE if FAST_QC else 50000

primary_layout = validate_slab_layout(primary_slabs)
print("\nPrimary layout checks:")
print(json.dumps(primary_layout, indent=2))

primary_stats = stream_stats(primary_slabs, max_cells=stream_cap)
print("\nPrimary streaming stats:")
print(json.dumps(primary_stats, indent=2))

primary_cont = continuity_ratio_on_sample(primary_slabs, sample_cells=sample_cont, k=6, seed=42)
print("\nPrimary continuity diagnostics:")
print(json.dumps(primary_cont, indent=2))

other_layout = None
other_stats = None
other_cont = None
ver_cmp = None

if COMPARE_OLD:
    other_layout = validate_slab_layout(other_slabs)
    other_stats = stream_stats(other_slabs, max_cells=stream_cap)
    other_cont = continuity_ratio_on_sample(other_slabs, sample_cells=sample_cont, k=6, seed=42)
    print("\nOther layout checks:")
    print(json.dumps(other_layout, indent=2))
    print("\nOther streaming stats:")
    print(json.dumps(other_stats, indent=2))
    print("\nOther continuity diagnostics:")
    print(json.dumps(other_cont, indent=2))

    if RUN_HEAVY_CROSS_VERSION and not FAST_QC:
        ver_cmp = compare_versions(primary_slabs, other_slabs)
        print("\nCross-version comparison:")
        print(json.dumps(ver_cmp, indent=2))

report = {
    'primary_dir': str(PRIMARY_DIR),
    'compare_old': bool(COMPARE_OLD),
    'fast_qc': bool(FAST_QC),
    'primary_layout': primary_layout,
    'primary_stats': primary_stats,
    'primary_continuity': primary_cont,
    'other_layout': other_layout,
    'other_stats': other_stats,
    'other_continuity': other_cont,
    'cross_version': ver_cmp,
}
with (OUT_DIR / 'tweb_validation_report.json').open('w', encoding='utf-8') as f:
    json.dump(report, f, indent=2)

if pd is not None:
    rows = [
        {
            'version': 'primary',
            'n_cells': primary_stats['n_cells'],
            'lambda1_mean': primary_stats['mean'][0],
            'lambda2_mean': primary_stats['mean'][1],
            'lambda3_mean': primary_stats['mean'][2],
            'lambda1_std': primary_stats['std'][0],
            'lambda2_std': primary_stats['std'][1],
            'lambda3_std': primary_stats['std'][2],
            'order_ok_fraction': primary_stats['order_ok_fraction'],
            'continuity_ratio_neighbor_over_random': primary_cont['continuity_ratio_neighbor_over_random'],
        }
    ]
    if COMPARE_OLD and other_stats is not None and other_cont is not None:
        rows.append(
            {
                'version': 'other',
                'n_cells': other_stats['n_cells'],
                'lambda1_mean': other_stats['mean'][0],
                'lambda2_mean': other_stats['mean'][1],
                'lambda3_mean': other_stats['mean'][2],
                'lambda1_std': other_stats['std'][0],
                'lambda2_std': other_stats['std'][1],
                'lambda3_std': other_stats['std'][2],
                'order_ok_fraction': other_stats['order_ok_fraction'],
                'continuity_ratio_neighbor_over_random': other_cont['continuity_ratio_neighbor_over_random'],
            }
        )
    display(pd.DataFrame(rows))

print(f"\nSaved report JSON: {OUT_DIR / 'new_tweb_validation_report.json'}")

In [ ]:
def sample_eigs_for_plot(slabs: list[SlabMeta], n: int = 400000, seed: int = 0) -> np.ndarray:
    rng = np.random.default_rng(seed)
    vols = np.array([(s.x_end - s.x_start) * s.ngrid * s.ngrid for s in slabs], dtype=np.float64)
    probs = vols / vols.sum()
    picks = rng.multinomial(n, probs)

    out = []
    for s, n_pick in zip(slabs, picks):
        if n_pick == 0:
            continue
        with np.load(s.path) as d:
            eig = np.asarray(d['eig_vals'], dtype=np.float32)
        nx = s.x_end - s.x_start
        ix = rng.integers(0, nx, size=n_pick)
        iy = rng.integers(0, s.ngrid, size=n_pick)
        iz = rng.integers(0, s.ngrid, size=n_pick)
        out.append(np.stack([eig[0, ix, iy, iz], eig[1, ix, iy, iz], eig[2, ix, iy, iz]], axis=1))
    return np.concatenate(out, axis=0)


def sample_cweb_for_plot(slabs: list[SlabMeta], n: int = 400000, seed: int = 1) -> np.ndarray:
    rng = np.random.default_rng(seed)
    vols = np.array([(s.x_end - s.x_start) * s.ngrid * s.ngrid for s in slabs], dtype=np.float64)
    probs = vols / vols.sum()
    picks = rng.multinomial(n, probs)

    out = []
    for s, n_pick in zip(slabs, picks):
        if n_pick == 0:
            continue
        with np.load(s.path) as d:
            cweb = np.asarray(d['cweb'])
        nx = s.x_end - s.x_start
        ix = rng.integers(0, nx, size=n_pick)
        iy = rng.integers(0, s.ngrid, size=n_pick)
        iz = rng.integers(0, s.ngrid, size=n_pick)
        out.append(cweb[ix, iy, iz])
    return np.concatenate(out, axis=0)


n_plot = FAST_SAMPLE_PLOT if FAST_QC else 400000
primary_eigs = sample_eigs_for_plot(primary_slabs, n=n_plot, seed=10)
primary_cls = sample_cweb_for_plot(primary_slabs, n=n_plot, seed=11)

other_eigs = sample_eigs_for_plot(other_slabs, n=n_plot, seed=10) if COMPARE_OLD else None
other_cls = sample_cweb_for_plot(other_slabs, n=n_plot, seed=11) if COMPARE_OLD else None

print(f"Sampled primary eigenvalues: {primary_eigs.shape}")
print(f"Sampled primary cweb labels: {primary_cls.shape}")
if COMPARE_OLD:
    print(f"Sampled other eigenvalues: {other_eigs.shape}")
    print(f"Sampled other cweb labels: {other_cls.shape}")

In [ ]:
# Histograms and CDFs for lambda1/lambda2/lambda3
fig, axes = plt.subplots(2, 3, figsize=(18, 9), constrained_layout=True)
labels = [r'$\lambda_1$', r'$\lambda_2$', r'$\lambda_3$']
colors = {'primary': '#5ec2ff', 'other': '#ff9b4a'}

for j in range(3):
    ax = axes[0, j]
    lo = primary_eigs[:, j].min()
    hi = primary_eigs[:, j].max()
    if COMPARE_OLD and other_eigs is not None:
        lo = min(lo, other_eigs[:, j].min())
        hi = max(hi, other_eigs[:, j].max())
    bins = np.linspace(lo, hi, 120)

    ax.hist(primary_eigs[:, j], bins=bins, density=True, alpha=0.55, color=colors['primary'], label='primary', edgecolor='none')
    if COMPARE_OLD and other_eigs is not None:
        ax.hist(other_eigs[:, j], bins=bins, density=True, alpha=0.45, color=colors['other'], label='other', edgecolor='none')

    ax.set_title(f'Distribution of {labels[j]}')
    ax.set_xlabel(labels[j])
    ax.set_ylabel('Density')
    ax.grid(alpha=0.2)
    if j == 0:
        ax.legend(frameon=False)

    ax = axes[1, j]
    xs = np.sort(primary_eigs[:, j])
    ys = np.linspace(0, 1, xs.size)
    ax.plot(xs, ys, color=colors['primary'], lw=2.0, label='primary')
    if COMPARE_OLD and other_eigs is not None:
        xs = np.sort(other_eigs[:, j])
        ys = np.linspace(0, 1, xs.size)
        ax.plot(xs, ys, color=colors['other'], lw=2.0, label='other')
    ax.set_title(f'CDF of {labels[j]}')
    ax.set_xlabel(labels[j])
    ax.set_ylabel('Cumulative probability')
    ax.grid(alpha=0.2)
    if j == 0:
        ax.legend(frameon=False)

fig.suptitle('T-Web Eigenvalue Distribution Sanity', fontsize=16)
out = OUT_DIR / 'new_eigen_hist_cdf_compare.png'
# fig.savefig(out, dpi=180, bbox_inches='tight')
plt.show()
print(f'Saved: {out}')

In [ ]:
# CWEB class fractions and ordering diagnostics
classes = np.arange(4)
class_names = ['void (0)', 'sheet (1)', 'filament (2)', 'knot (3)']

primary_frac = np.array([(primary_cls == c).mean() for c in classes])
other_frac = np.array([(other_cls == c).mean() for c in classes]) if (COMPARE_OLD and other_cls is not None) else None

fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)

x = np.arange(4)
w = 0.38
if COMPARE_OLD and other_frac is not None:
    axes[0].bar(x - w/2, primary_frac, width=w, color='#5ec2ff', alpha=0.9, label='primary')
    axes[0].bar(x + w/2, other_frac, width=w, color='#ff9b4a', alpha=0.9, label='other')
else:
    axes[0].bar(x, primary_frac, width=0.7, color='#5ec2ff', alpha=0.9, label='primary')

axes[0].set_xticks(x)
axes[0].set_xticklabels(class_names, rotation=20, ha='right')
axes[0].set_ylabel('Fraction')
axes[0].set_xlabel('CWEB class')
axes[0].set_title('CWEB Class Fractions')
axes[0].legend(frameon=False)
axes[0].grid(axis='y', alpha=0.2)

order_p_12 = float(np.mean(primary_eigs[:, 0] > primary_eigs[:, 1]))
order_p_23 = float(np.mean(primary_eigs[:, 1] > primary_eigs[:, 2]))

vals = [order_p_12, order_p_23]
labels = ['primary: l1>l2', 'primary: l2>l3']
colors = ['#5ec2ff', '#5ec2ff']

if COMPARE_OLD and other_eigs is not None:
    order_o_12 = float(np.mean(other_eigs[:, 0] > other_eigs[:, 1]))
    order_o_23 = float(np.mean(other_eigs[:, 1] > other_eigs[:, 2]))
    vals.extend([order_o_12, order_o_23])
    labels.extend(['other: l1>l2', 'other: l2>l3'])
    colors.extend(['#ff9b4a', '#ff9b4a'])

axes[1].bar(np.arange(len(vals)), vals, color=colors, alpha=0.9)
axes[1].set_xticks(np.arange(len(vals)))
axes[1].set_xticklabels(labels, rotation=15, ha='right')
axes[1].set_ylabel('Violation fraction')
axes[1].set_xlabel('Ordering check')
axes[1].set_title('Eigenvalue Ordering Violations')
axes[1].grid(axis='y', alpha=0.2)

out = OUT_DIR / 'new_cweb_and_ordering_compare.png'
fig.savefig(out, dpi=180, bbox_inches='tight')
plt.show()
print(f'Saved: {out}')

In [ ]:
# Optional heavy cross-version eig scatter diagnostics
if COMPARE_OLD and RUN_HEAVY_CROSS_VERSION and (not FAST_QC):
    rng = np.random.default_rng(123)
    other_by_rank = {s.rank: s for s in other_slabs}

    pairs_primary = []
    pairs_other = []
    max_per_slab = 20000

    for ps in primary_slabs:
        os = other_by_rank.get(ps.rank)
        if os is None:
            continue
        with np.load(ps.path) as dp, np.load(os.path) as do:
            ep = np.asarray(dp['eig_vals'], dtype=np.float32)
            eo = np.asarray(do['eig_vals'], dtype=np.float32)
        if ep.shape != eo.shape:
            continue

        nx, ny, nz = ep.shape[1], ep.shape[2], ep.shape[3]
        n = min(max_per_slab, nx * ny * nz)
        ix = rng.integers(0, nx, size=n)
        iy = rng.integers(0, ny, size=n)
        iz = rng.integers(0, nz, size=n)

        lam_primary = np.stack([ep[0, ix, iy, iz], ep[1, ix, iy, iz], ep[2, ix, iy, iz]], axis=1)
        lam_other = np.stack([eo[0, ix, iy, iz], eo[1, ix, iy, iz], eo[2, ix, iy, iz]], axis=1)
        pairs_primary.append(lam_primary)
        pairs_other.append(lam_other)

    pairs_primary = np.concatenate(pairs_primary, axis=0)
    pairs_other = np.concatenate(pairs_other, axis=0)
    diff = pairs_primary - pairs_other

    fig, axes = plt.subplots(2, 3, figsize=(16, 9), constrained_layout=True)
    labels = [r'$\lambda_1$', r'$\lambda_2$', r'$\lambda_3$']

    for j in range(3):
        ax = axes[0, j]
        x = pairs_other[:, j]
        y = pairs_primary[:, j]
        n_show = min(150000, x.size)
        sel = rng.choice(x.size, size=n_show, replace=False)
        ax.scatter(x[sel], y[sel], s=2, alpha=0.12, color='#8be28b')
        lo = min(x.min(), y.min())
        hi = max(x.max(), y.max())
        ax.plot([lo, hi], [lo, hi], '--', color='white', lw=1.0)
        corr = float(np.corrcoef(x, y)[0, 1])
        ax.set_title(f'{labels[j]} other vs primary (corr={corr:.4f})')
        ax.set_xlabel(f'other {labels[j]}')
        ax.set_ylabel(f'primary {labels[j]}')

        ax = axes[1, j]
        ax.hist(diff[:, j], bins=140, density=True, color='#ff5f6d', alpha=0.8)
        ax.set_title(f'Residual: primary - other for {labels[j]}')
        ax.set_xlabel(f'primary - other {labels[j]}')
        ax.set_ylabel('Density')
        ax.grid(alpha=0.2)

    fig.suptitle('Cross-Version Eigenvalue Agreement (Matched Cell Samples)', fontsize=16)
    out = OUT_DIR / 'cross_version_scatter_residuals.png'
    fig.savefig(out, dpi=180, bbox_inches='tight')
    plt.show()
    print(f'Saved: {out}')
else:
    print('Skipping heavy cross-version scatter/residual diagnostics (FAST_QC or COMPARE_OLD disabled).')

In [ ]:
# Summary figure for quick pass/fail interpretation
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8), constrained_layout=True)

# 1) Means and stds
x = np.arange(3)
means_primary = np.array(primary_stats['mean'])
std_primary = np.array(primary_stats['std'])

if COMPARE_OLD and (other_stats is not None):
    means_other = np.array(other_stats['mean'])
    std_other = np.array(other_stats['std'])
    axes[0].errorbar(x - 0.04, means_primary, yerr=std_primary, fmt='o', color='#5ec2ff', label='primary', capsize=4)
    axes[0].errorbar(x + 0.04, means_other, yerr=std_other, fmt='o', color='#ff9b4a', label='other', capsize=4)
    axes[0].legend(frameon=False)
else:
    axes[0].errorbar(x, means_primary, yerr=std_primary, fmt='o', color='#5ec2ff', label='primary', capsize=4)

axes[0].set_xticks(x)
axes[0].set_xticklabels([r'$\lambda_1$', r'$\lambda_2$', r'$\lambda_3$'])
axes[0].set_title('Moments by Eigenvalue')
axes[0].set_xlabel('Eigenvalue')
axes[0].set_ylabel('Mean +/- std')
axes[0].grid(alpha=0.2)

# 2) Continuity ratio
if COMPARE_OLD and (other_cont is not None):
    cont_vals = [primary_cont['continuity_ratio_neighbor_over_random'], other_cont['continuity_ratio_neighbor_over_random']]
    axes[1].bar([0, 1], cont_vals, color=['#5ec2ff', '#ff9b4a'], alpha=0.9)
    axes[1].set_xticks([0, 1])
    axes[1].set_xticklabels(['primary', 'other'])
else:
    cont_vals = [primary_cont['continuity_ratio_neighbor_over_random']]
    axes[1].bar([0], cont_vals, color=['#5ec2ff'], alpha=0.9)
    axes[1].set_xticks([0])
    axes[1].set_xticklabels(['primary'])

axes[1].axhline(1.0, color='white', ls='--', lw=1.0)
axes[1].set_xlabel('Version')
axes[1].set_ylabel('Neighbor/random absdiff ratio')
axes[1].set_title('Local Continuity (lower is better)')
axes[1].grid(axis='y', alpha=0.2)

# 3) Cross-version MAE/RMSE (optional)
if isinstance(ver_cmp, dict) and ver_cmp.get('n_cells_compared', 0) > 0:
    mae = np.array(ver_cmp['mae_lambda'])
    rmse = np.array(ver_cmp['rmse_lambda'])
    w = 0.35
    axes[2].bar(x - w/2, mae, width=w, color='#ffd166', label='MAE')
    axes[2].bar(x + w/2, rmse, width=w, color='#ef476f', label='RMSE')
    axes[2].set_xticks(x)
    axes[2].set_xticklabels([r'$\lambda_1$', r'$\lambda_2$', r'$\lambda_3$'])
    axes[2].set_xlabel('Eigenvalue')
    axes[2].set_ylabel('Error')
    axes[2].set_title(f"Other vs Primary Error (CWEB match={ver_cmp['cweb_match_fraction']:.4f})")
    axes[2].legend(frameon=False)
    axes[2].grid(axis='y', alpha=0.2)
else:
    axes[2].text(0.5, 0.5, 'Cross-version metrics disabled in FAST_QC', ha='center', va='center')
    axes[2].set_axis_off()

fig.suptitle('T-Web Validation Quick Summary', fontsize=16)
out = OUT_DIR / 'new_tweb_validation_summary.png'
fig.savefig(out, dpi=180, bbox_inches='tight')
plt.show()
print(f'Saved: {out}')

## Interpretation guide

- **Ordering violations** should be near zero if eigenvalues are sorted correctly.
- **Continuity ratio neighbor/random**:
  - `< 1.0`: local structure is smoother than random (expected for coherent fields)
  - `~ 1.0`: weak local coherence
  - `> 1.0`: potentially noisy/inconsistent labels
- **Class fractions** should look physically plausible and stable under reruns with same config.
- **Cross-version MAE/RMSE** and **CWEB match fraction** are optional checks and disabled by default in `FAST_QC` mode.

### Recommended usage

- Keep `FAST_QC=True` and `COMPARE_OLD=False` for rapid physical sanity checks.
- Only set `COMPARE_OLD=True` and `RUN_HEAVY_CROSS_VERSION=True` when you explicitly need old-vs-new deep diagnostics.
- If runtime is still long, reduce `FAST_MAX_STREAM_CELLS`, `FAST_SAMPLE_PLOT`, and `FAST_CONTINUITY_SAMPLE`.

## Optional: Graph Metric vs Eigenvalue Signal Audit

This section computes:

- **Pearson correlation** between each graph metric and each eigenvalue target (`LAMBDA1/2/3`)
- **Mutual information** between each graph metric and each eigenvalue target

It uses the Abacus graph metric parquet and source FITS catalog from the GNN metadata file by default, with optional row subsampling for speed.

Set `CATALOG_OVERRIDE` in the config cell to use an alternative annotated catalog (for example your new `..._with_tweb_eigs.fits`).

In [ ]:
# --- Self-contained: graph metrics vs eigenvalues/CWEB (no prior cells required) ---
from pathlib import Path
from datetime import datetime
import json as _json
import numpy as _np
import matplotlib.pyplot as plt

try:
    import pandas as _pd
except Exception as e:
    raise RuntimeError('pandas is required for this section.') from e

try:
    import fitsio
except Exception as e:
    raise RuntimeError('fitsio is required for this section.') from e

# Standalone output directory (used for CSV/PNG outputs)
if 'OUT_DIR' not in globals() or globals().get('OUT_DIR') is None:
    OUT_DIR = Path('/pscratch/sd/d/dkololgi/abacus/alignment_diagnostics') / f"graph_corr_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    print(f"OUT_DIR (auto): {OUT_DIR}")

# --- Config ---
# CutSky triangulation in observed-space (RA, Dec, Z) — old baseline:
# GNN_METADATA_PATH = Path('/pscratch/sd/d/dkololgi/abacus/graph_constructions/abacus_alpha_cugraph_gnn_metadata.json')
# Same galaxy rows; Delaunay / metrics on halo box-frame xyz (commensurate with T-Web labels):
GNN_METADATA_PATH = Path('/pscratch/sd/d/dkololgi/abacus/graph_constructions/abacus_mock_alpha_23032026_cugraph_gnn_metadata.json')
#Path('/pscratch/sd/d/dkololgi/abacus/graph_constructions/abacus_alpha_boxframe_cugraph_gnn_metadata.json')
CATALOG_OVERRIDE = '/pscratch/sd/d/dkololgi/abacus/mocks_with_eigs/cutsky_BGS_z0.200_AbacusSummit_base_c000_ph000_with_tweb_eigs_ng2048_rs4_v2.fits'
# Alternate catalog (comment / swap if needed):
# CATALOG_OVERRIDE = '/pscratch/sd/d/dkololgi/abacus/mocks_with_eigs/cutsky_BGS_z0.200_AbacusSummit_base_c000_ph000_with_tweb_eigs_ng512_rs24.fits'

FEATURE_COLUMNS = ['Degree', 'Clustering', 'Density', 'Neigh Density', 'I_eig1', 'I_eig2', 'I_eig3']
# If True: correlate / MI graph metrics vs discrete CWEB class {0,1,2,3}.
# If False: vs continuous eigenvalues (lambda1, lambda2, lambda3).
USE_CWEB_LABELS = False
MAX_ROWS = 1_000_000  # Row cap for speed; set None for full dataset
SEED = 42
RUN_MUTUAL_INFO = True
MI_MAX_ROWS = 300_000  # MI is slower; subsample for throughput


def _resolve_col(dtype_names, candidates):
    names = {n.upper(): n for n in dtype_names}
    for c in candidates:
        k = c.upper()
        if k in names:
            return names[k]
    raise KeyError(f'None of candidate columns {list(candidates)} found. Sample names: {list(dtype_names)[:20]}')


def _apply_optional_y1y5_filter(table, x, y):
    """Apply the same row selection as graph construction (when columns exist).

    Graph build (`build_abacus_graph.py` catalog mode) uses:
    - (IN_Y1 == 1) OR (IN_Y5 == 1)
    - BOX_INDEX != -1

    The annotated FITS typically contains *all* CutSky rows (63M), with T-Web values
    NaN/255 for unmatched rows. Feature tables only exist for the selected subset.
    """
    names_upper = {n.upper(): n for n in table.dtype.names}

    # Accept both IN_Y* and Y* naming variants.
    in_y1 = names_upper.get('IN_Y1') or names_upper.get('Y1')
    in_y5 = names_upper.get('IN_Y5') or names_upper.get('Y5')
    box_i = names_upper.get('BOX_INDEX') or names_upper.get('BOXINDEX')

    mask = _np.ones(len(table), dtype=bool)

    # Optional Y1/Y5 filter (some catalogs might not have it).
    if in_y1 is not None or in_y5 is not None:
        m = _np.zeros(len(table), dtype=bool)
        if in_y1 is not None:
            m |= (_np.asarray(table[in_y1]) == 1)
        if in_y5 is not None:
            m |= (_np.asarray(table[in_y5]) == 1)
        if not m.any():
            m[:] = True
        mask &= m
    else:
        print('No Y1/Y5 columns found; skipping Y1/Y5 filter.')

    # Optional BOX_INDEX cut (crucial for matching the ~18M graph build).
    if box_i is not None:
        mask &= (_np.asarray(table[box_i]) != -1)
    else:
        print('No BOX_INDEX column found; skipping BOX_INDEX != -1 filter.')

    y_f = y[mask]

    # Case A: x and y are both full-catalog aligned.
    if x.shape[0] == y.shape[0]:
        return x[mask], y_f

    # Case B: x already built on the selected subset.
    if x.shape[0] == y_f.shape[0]:
        return x, y_f

    print(
        'Warning: could not safely apply (Y1|Y5, BOX_INDEX) filter to align rows. '
        f'x rows={x.shape[0]:,}, y rows={y.shape[0]:,}, y_filtered rows={y_f.shape[0]:,}'
    )
    return x, y


# --- Load features + targets ---
import json as _json
import numpy as _np

try:
    import pandas as _pd
except Exception as e:
    raise RuntimeError('pandas is required for this section.') from e

try:
    import fitsio
except Exception as e:
    raise RuntimeError('fitsio is required for this section.') from e

with GNN_METADATA_PATH.open('r', encoding='utf-8') as f:
    gnn_meta = _json.load(f)


def _resolve_source_catalog_path(gnn_meta_dict, override):
    if override:
        return Path(override).expanduser().resolve()

    # Prefer direct source path in graph metadata if present.
    direct_candidates = [
        gnn_meta_dict.get('source_path'),
        gnn_meta_dict.get('catalog_path'),
    ]
    for c in direct_candidates:
        if c:
            return Path(c).expanduser().resolve()

    # Some schemas use nested inputs.
    inputs = gnn_meta_dict.get('inputs', {})
    if isinstance(inputs, dict):
        for key in ('source_catalog', 'catalog_path', 'source_path'):
            if inputs.get(key):
                return Path(inputs[key]).expanduser().resolve()

    # Fallback: follow input_metadata_path and read source_path there.
    inp_meta = gnn_meta_dict.get('input_metadata_path')
    if inp_meta:
        inp_meta_path = Path(inp_meta).expanduser().resolve()
        with inp_meta_path.open('r', encoding='utf-8') as f:
            inp_meta_dict = _json.load(f)
        for key in ('source_path', 'catalog_path', 'source_catalog'):
            if inp_meta_dict.get(key):
                return Path(inp_meta_dict[key]).expanduser().resolve()

    raise KeyError(
        'Could not resolve source catalog path. Set CATALOG_OVERRIDE or provide one of '
        '[source_path, inputs.source_catalog, input_metadata_path->source_path] in metadata.'
    )


node_parquet = Path(gnn_meta['outputs']['node_features']).expanduser().resolve()
source_catalog = _resolve_source_catalog_path(gnn_meta, CATALOG_OVERRIDE)

print(f'Loading features: {node_parquet}')
node_df = _pd.read_parquet(node_parquet, columns=FEATURE_COLUMNS)
x = node_df.to_numpy(dtype=_np.float64)

print(f'Loading targets: {source_catalog}')
tab = fitsio.read(str(source_catalog))

if USE_CWEB_LABELS:
    cweb_col = _resolve_col(tab.dtype.names, ('CWEB', 'CWEB_CLASS', 'ENV', 'TWEB'))
    y = _np.asarray(tab[cweb_col], dtype=_np.float64).reshape(-1, 1)
    target_mode = 'cweb'
    target_columns = ['cweb']
    print(f'Target mode: CWEB (column {cweb_col}), unique values: {_np.unique(y)}')
else:
    l1_col = _resolve_col(tab.dtype.names, ('LAMBDA1', 'L1', 'EIG1', 'LAM1', 'LAMBDA_1'))
    l2_col = _resolve_col(tab.dtype.names, ('LAMBDA2', 'L2', 'EIG2', 'LAM2', 'LAMBDA_2'))
    l3_col = _resolve_col(tab.dtype.names, ('LAMBDA3', 'L3', 'EIG3', 'LAM3', 'LAMBDA_3'))
    y = _np.stack([
        _np.asarray(tab[l1_col], dtype=_np.float64),
        _np.asarray(tab[l2_col], dtype=_np.float64),
        _np.asarray(tab[l3_col], dtype=_np.float64),
    ], axis=1)
    target_mode = 'eigenvalues'
    target_columns = ['lambda1', 'lambda2', 'lambda3']

# Apply the same optional Y1/Y5 selection used in graph construction before alignment checks.
x, y = _apply_optional_y1y5_filter(tab, x, y)
print(f'Rows after optional Y1/Y5 filter: {x.shape[0]:,}')

if x.shape[0] != y.shape[0]:
    raise ValueError(
        f'Row mismatch after filtering: features={x.shape[0]:,}, targets={y.shape[0]:,}. '
        'This usually means the feature parquet and FITS catalog were not built from identical selections.'
    )

# Optional row cap for speed.
rng = _np.random.default_rng(SEED)
if MAX_ROWS is not None and x.shape[0] > int(MAX_ROWS):
    idx = rng.choice(x.shape[0], size=int(MAX_ROWS), replace=False)
    x = x[idx]
    y = y[idx]
    print(f'Using subsample for correlations: {x.shape[0]:,} rows')

# --- Pearson correlation ---
xc = x - x.mean(axis=0, keepdims=True)
yc = y - y.mean(axis=0, keepdims=True)
num = xc.T @ yc
xnorm = _np.sqrt(_np.sum(xc * xc, axis=0, keepdims=True)).T  # [F,1]
ynorm = _np.sqrt(_np.sum(yc * yc, axis=0, keepdims=True))    # [1,K]
pearson = num / _np.maximum(xnorm * ynorm, 1e-12)

pearson_df = _pd.DataFrame(pearson, index=FEATURE_COLUMNS, columns=target_columns)

if USE_CWEB_LABELS:
    print('\nPearson correlation (feature vs CWEB class, ordinal treatment):')
else:
    print('\nPearson correlation (feature vs eigenvalue):')
display(pearson_df)

pearson_out = OUT_DIR / f'graph_metric_pearson_vs_{target_mode}.csv'
pearson_df.to_csv(pearson_out, index=True)
print(f'Saved: {pearson_out}')

# --- Mutual information ---
mi_df = None
if RUN_MUTUAL_INFO:
    try:
        from sklearn.feature_selection import mutual_info_regression, mutual_info_classif

        xm = x
        ym = y
        if MI_MAX_ROWS is not None and x.shape[0] > int(MI_MAX_ROWS):
            idx_mi = rng.choice(x.shape[0], size=int(MI_MAX_ROWS), replace=False)
            xm = x[idx_mi]
            ym = y[idx_mi]
            print(f'Using subsample for MI: {xm.shape[0]:,} rows')

        if USE_CWEB_LABELS:
            yc_flat = _np.asarray(ym.ravel(), dtype=_np.int64)
            mi_vec = mutual_info_classif(
                xm,
                yc_flat,
                discrete_features=False,
                discrete_target=True,
                random_state=SEED,
            )
            mi_df = _pd.DataFrame(mi_vec.reshape(-1, 1), index=FEATURE_COLUMNS, columns=['cweb'])
            print('\nMutual information (feature vs CWEB class):')
        else:
            mi = _np.zeros((xm.shape[1], ym.shape[1]), dtype=_np.float64)
            for j in range(ym.shape[1]):
                mi[:, j] = mutual_info_regression(xm, ym[:, j], random_state=SEED)
            mi_df = _pd.DataFrame(mi, index=FEATURE_COLUMNS, columns=target_columns)
            print('\nMutual information (feature vs eigenvalue):')
        display(mi_df)

        mi_out = OUT_DIR / f'graph_metric_mutual_info_vs_{target_mode}.csv'
        mi_df.to_csv(mi_out, index=True)
        print(f'Saved: {mi_out}')
    except Exception as e:
        print(f'Skipping MI due to missing dependency or runtime error: {e}')

# --- Plots ---
pearson_title = (
    'Pearson: Graph Metrics vs CWEB (ordinal r)'
    if USE_CWEB_LABELS
    else 'Pearson Correlation: Graph Metrics vs Eigenvalues'
)
fig, ax = plt.subplots(figsize=(7, 5), constrained_layout=True)
im = ax.imshow(pearson_df.values, aspect='auto', cmap='coolwarm', vmin=-1.0, vmax=1.0)
ax.set_title(pearson_title)
ax.set_xlabel('Target')
ax.set_ylabel('Graph metric feature')
ax.set_xticks(_np.arange(pearson_df.shape[1]))
ax.set_xticklabels(list(pearson_df.columns))
ax.set_yticks(_np.arange(pearson_df.shape[0]))
ax.set_yticklabels(list(pearson_df.index))
for i in range(pearson_df.shape[0]):
    for j in range(pearson_df.shape[1]):
        ax.text(j, i, f'{pearson_df.values[i, j]:.3f}', ha='center', va='center', fontsize=8)
cb = fig.colorbar(im, ax=ax)
cb.set_label('Pearson r')
out = OUT_DIR / f'graph_metric_pearson_heatmap_{target_mode}.png'
fig.savefig(out, dpi=180, bbox_inches='tight')
plt.show()
print(f'Saved: {out}')

if mi_df is not None:
    mi_title = (
        'Mutual Information: Graph Metrics vs CWEB'
        if USE_CWEB_LABELS
        else 'Mutual Information: Graph Metrics vs Eigenvalues'
    )
    fig, ax = plt.subplots(figsize=(7, 5), constrained_layout=True)
    im = ax.imshow(mi_df.values, aspect='auto', cmap='viridis')
    ax.set_title(mi_title)
    ax.set_xlabel('Target')
    ax.set_ylabel('Graph metric feature')
    ax.set_xticks(_np.arange(mi_df.shape[1]))
    ax.set_xticklabels(list(mi_df.columns))
    ax.set_yticks(_np.arange(mi_df.shape[0]))
    ax.set_yticklabels(list(mi_df.index))
    for i in range(mi_df.shape[0]):
        for j in range(mi_df.shape[1]):
            ax.text(j, i, f'{mi_df.values[i, j]:.3f}', ha='center', va='center', fontsize=8)
    cb = fig.colorbar(im, ax=ax)
    cb.set_label('Mutual information')
    out = OUT_DIR / f'graph_metric_mutual_info_heatmap_{target_mode}.png'
    fig.savefig(out, dpi=180, bbox_inches='tight')
    plt.show()
    print(f'Saved: {out}')

## Additional Controls: Position Signal, Richer Baselines, and CWEB Classification

These checks help diagnose whether low graph-metric/eigenvalue signal is due to feature compression vs target/domain mismatch.

1. **Position-only control** (`RA`, `DEC`, `Z`, optionally `Z_COSMO`) against eigenvalues.
2. **Richer tabular regression baseline** comparing:
   - graph metrics only,
   - position-only,
   - graph + position.
3. **CWEB classification baseline** using graph and position features.

All runs use capped random subsets for speed.

In [ ]:
# --- Control config ---
CONTROL_MAX_ROWS = 300_000
CONTROL_TEST_FRAC = 0.2
CONTROL_SEED = 42
RUN_CONTROL_MI = True

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, balanced_accuracy_score
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier

# Resolve catalog path consistently with earlier section.
catalog_for_controls = Path(CATALOG_OVERRIDE).expanduser().resolve() if CATALOG_OVERRIDE else Path(source_catalog).expanduser().resolve()
print(f"Control catalog: {catalog_for_controls}")

# Load only needed columns for controls.
base_cols = ["RA", "DEC", "Z", "Z_COSMO", "CWEB", "IN_Y1", "IN_Y5", l1_col, l2_col, l3_col]
with fitsio.FITS(str(catalog_for_controls)) as f:
    all_cols = set(f[1].get_colnames())
use_cols = [c for c in base_cols if c in all_cols]
ctrl_tab = fitsio.read(str(catalog_for_controls), columns=use_cols)

# Build Y1/Y5 filter consistent with graph construction logic.
names_upper = {n.upper(): n for n in ctrl_tab.dtype.names}
in_y1 = names_upper.get("IN_Y1") or names_upper.get("Y1")
in_y5 = names_upper.get("IN_Y5") or names_upper.get("Y5")
mask = np.ones(len(ctrl_tab), dtype=bool)
if in_y1 is not None or in_y5 is not None:
    mask = np.zeros(len(ctrl_tab), dtype=bool)
    if in_y1 is not None:
        mask |= (np.asarray(ctrl_tab[in_y1]) == 1)
    if in_y5 is not None:
        mask |= (np.asarray(ctrl_tab[in_y5]) == 1)

ra = np.asarray(ctrl_tab[names_upper.get("RA", "RA")], dtype=np.float64)[mask]
dec = np.asarray(ctrl_tab[names_upper.get("DEC", "DEC")], dtype=np.float64)[mask]
zobs = np.asarray(ctrl_tab[names_upper.get("Z", "Z")], dtype=np.float64)[mask]
zc = np.asarray(ctrl_tab[names_upper.get("Z_COSMO", "Z_COSMO")], dtype=np.float64)[mask] if "Z_COSMO" in names_upper else np.full_like(zobs, np.nan)

y_all = np.stack([
    np.asarray(ctrl_tab[l1_col], dtype=np.float64)[mask],
    np.asarray(ctrl_tab[l2_col], dtype=np.float64)[mask],
    np.asarray(ctrl_tab[l3_col], dtype=np.float64)[mask],
], axis=1)

# Graph features from parquet, same columns as prior section.
node_parquet = Path(gnn_meta['outputs']['node_features']).expanduser().resolve()
x_all = pd.read_parquet(node_parquet, columns=FEATURE_COLUMNS).to_numpy(dtype=np.float64)
if x_all.shape[0] != y_all.shape[0]:
    raise ValueError(f"Control row mismatch after filtering: features={x_all.shape[0]:,}, targets={y_all.shape[0]:,}")

# Shared subsample for all controls.
rng = np.random.default_rng(CONTROL_SEED)
n_tot = x_all.shape[0]
if CONTROL_MAX_ROWS is not None and n_tot > int(CONTROL_MAX_ROWS):
    idx = rng.choice(n_tot, size=int(CONTROL_MAX_ROWS), replace=False)
    x_c = x_all[idx]
    y_c = y_all[idx]
    ra_c, dec_c, zobs_c, zc_c = ra[idx], dec[idx], zobs[idx], zc[idx]
    cweb_c = np.asarray(ctrl_tab[names_upper["CWEB"]], dtype=np.int32)[mask][idx] if "CWEB" in names_upper else None
else:
    x_c = x_all
    y_c = y_all
    ra_c, dec_c, zobs_c, zc_c = ra, dec, zobs, zc
    cweb_c = np.asarray(ctrl_tab[names_upper["CWEB"]], dtype=np.int32)[mask] if "CWEB" in names_upper else None

pos_c = np.column_stack([ra_c, dec_c, zobs_c, zc_c])

print(f"Control sample rows: {x_c.shape[0]:,}")

# 1) Position-only correlation / MI against eigenvalues.
def _pearson_mat(a, b):
    ac = a - a.mean(axis=0, keepdims=True)
    bc = b - b.mean(axis=0, keepdims=True)
    num = ac.T @ bc
    den = np.sqrt(np.sum(ac * ac, axis=0, keepdims=True)).T * np.sqrt(np.sum(bc * bc, axis=0, keepdims=True))
    return num / np.maximum(den, 1e-12)

pos_names = ["RA", "DEC", "Z", "Z_COSMO"]
pos_pear = _pearson_mat(pos_c, y_c)
pos_pear_df = pd.DataFrame(pos_pear, index=pos_names, columns=["lambda1", "lambda2", "lambda3"])
print("\nPosition-only Pearson vs eigenvalues:")
display(pos_pear_df)

if RUN_CONTROL_MI:
    try:
        from sklearn.feature_selection import mutual_info_regression
        mi_pos = np.column_stack([
            mutual_info_regression(pos_c, y_c[:, j], random_state=CONTROL_SEED)
            for j in range(y_c.shape[1])
        ])
        mi_pos_df = pd.DataFrame(mi_pos, index=pos_names, columns=["lambda1", "lambda2", "lambda3"])
        print("\nPosition-only MI vs eigenvalues:")
        display(mi_pos_df)
    except Exception as e:
        print(f"Skipping position-only MI: {e}")

# 2) Richer tabular regression baseline.
X_graph = x_c
X_pos = pos_c
X_both = np.column_stack([x_c, pos_c])

Xg_tr, Xg_te, yg_tr, yg_te = train_test_split(X_graph, y_c, test_size=CONTROL_TEST_FRAC, random_state=CONTROL_SEED)
Xp_tr, Xp_te, yp_tr, yp_te = train_test_split(X_pos, y_c, test_size=CONTROL_TEST_FRAC, random_state=CONTROL_SEED)
Xb_tr, Xb_te, yb_tr, yb_te = train_test_split(X_both, y_c, test_size=CONTROL_TEST_FRAC, random_state=CONTROL_SEED)

reg = RandomForestRegressor(
    n_estimators=300,
    max_depth=24,
    min_samples_leaf=4,
    random_state=CONTROL_SEED,
    n_jobs=-1,
)

reg.fit(Xg_tr, yg_tr)
yg_hat = reg.predict(Xg_te)
r2_graph = r2_score(yg_te, yg_hat, multioutput='raw_values')

reg.fit(Xp_tr, yp_tr)
yp_hat = reg.predict(Xp_te)
r2_pos = r2_score(yp_te, yp_hat, multioutput='raw_values')

reg.fit(Xb_tr, yb_tr)
yb_hat = reg.predict(Xb_te)
r2_both = r2_score(yb_te, yb_hat, multioutput='raw_values')

r2_df = pd.DataFrame(
    {
        "graph_only": r2_graph,
        "position_only": r2_pos,
        "graph_plus_position": r2_both,
    },
    index=["lambda1", "lambda2", "lambda3"],
)
print("\nRegression R2 control (RandomForestRegressor):")
display(r2_df)

# 3) CWEB classification baseline.
if cweb_c is not None:
    Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(X_graph, cweb_c, test_size=CONTROL_TEST_FRAC, random_state=CONTROL_SEED, stratify=cweb_c)
    clf = RandomForestClassifier(
        n_estimators=300,
        max_depth=24,
        min_samples_leaf=4,
        random_state=CONTROL_SEED,
        n_jobs=-1,
    )
    clf.fit(Xc_tr, yc_tr)
    yc_hat_graph = clf.predict(Xc_te)
    bal_graph = balanced_accuracy_score(yc_te, yc_hat_graph)

    Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(X_pos, cweb_c, test_size=CONTROL_TEST_FRAC, random_state=CONTROL_SEED, stratify=cweb_c)
    clf.fit(Xc_tr, yc_tr)
    yc_hat_pos = clf.predict(Xc_te)
    bal_pos = balanced_accuracy_score(yc_te, yc_hat_pos)

    Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(X_both, cweb_c, test_size=CONTROL_TEST_FRAC, random_state=CONTROL_SEED, stratify=cweb_c)
    clf.fit(Xc_tr, yc_tr)
    yc_hat_both = clf.predict(Xc_te)
    bal_both = balanced_accuracy_score(yc_te, yc_hat_both)

    cweb_df = pd.DataFrame(
        {
            "balanced_accuracy": [bal_graph, bal_pos, bal_both],
        },
        index=["graph_only", "position_only", "graph_plus_position"],
    )
    print("\nCWEB classification control (RandomForestClassifier):")
    display(cweb_df)
else:
    print("CWEB column not present in this catalog; skipping classification control.")

# Save control outputs for reproducibility.
out_ctrl = OUT_DIR / "control_baselines_summary.json"
payload = {
    "control_rows": int(x_c.shape[0]),
    "r2_graph_only": r2_graph.tolist(),
    "r2_position_only": r2_pos.tolist(),
    "r2_graph_plus_position": r2_both.tolist(),
}
if cweb_c is not None:
    payload["cweb_balanced_accuracy"] = {
        "graph_only": float(bal_graph),
        "position_only": float(bal_pos),
        "graph_plus_position": float(bal_both),
    }
import json
with out_ctrl.open("w", encoding="utf-8") as f:
    json.dump(payload, f, indent=2)
print(f"Saved control summary: {out_ctrl}")

## 2D Cosmic-Web Slice Visualization

This section visualizes discrete T-Web environments (`0,1,2,3`) on global x-slices to sanity-check whether filamentary/wall/knot structures look reasonable.

- Uses threshold `0.2` for eigenvalue-count classification.
- Works for any `ngrid` because slices are selected by global x-fraction (not fixed indices).

In [ ]:
# 2D CWEB slices from slab outputs (global x-index aware)
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch

SLICE_THRESHOLD = 0.2  # requested threshold
SLICE_X_FRACTIONS = [0.2, 0.5, 0.8]

# Always reload from current PRIMARY_DIR to avoid stale notebook state.
primary_slabs = discover_slabs(PRIMARY_DIR)

ngrid = int(primary_slabs[0].ngrid)
boxsize = float(primary_slabs[0].boxsize)
print(f"Using PRIMARY_DIR={PRIMARY_DIR}")
print(f"Detected ngrid={ngrid}, boxsize={boxsize}")


def _load_global_x_slice(slabs: list[SlabMeta], global_ix: int):
    for s in slabs:
        if s.x_start <= global_ix < s.x_end:
            local_ix = int(global_ix - s.x_start)
            with np.load(s.path) as d:
                cweb = np.asarray(d['cweb'][local_ix, :, :], dtype=np.uint8)
                eig = np.asarray(d['eig_vals'][:, local_ix, :, :], dtype=np.float32)
            return cweb, eig, s
    raise ValueError(f'Global ix={global_ix} not covered by any slab.')


def _class_from_eigs(eig_slice: np.ndarray, threshold: float) -> np.ndarray:
    # eig_slice shape: [3, ny, nz], already ordered lambda1<=lambda2<=lambda3
    n_ge = np.sum(eig_slice >= threshold, axis=0)
    return np.asarray(n_ge, dtype=np.uint8)


ix_values = sorted({int(np.clip(round(f * (ngrid - 1)), 0, ngrid - 1)) for f in SLICE_X_FRACTIONS})

# Discrete colormap for CWEB classes.
class_colors = ['#1f77b4', '#2ca02c', '#ff7f0e', '#d62728']  # 0..3
class_labels = ['0: void', '1: sheet/wall', '2: filament', '3: knot']
cmap = ListedColormap(class_colors)
norm = BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5], cmap.N)

fig, axes = plt.subplots(len(ix_values), 2, figsize=(12, 4 * len(ix_values)), constrained_layout=True)
if len(ix_values) == 1:
    axes = np.asarray([axes])

match_fracs = []
for r, ixg in enumerate(ix_values):
    cweb_slice, eig_slice, slab = _load_global_x_slice(primary_slabs, ixg)
    cweb_from_eig = _class_from_eigs(eig_slice, threshold=SLICE_THRESHOLD)
    match_frac = float(np.mean(cweb_slice == cweb_from_eig))
    match_fracs.append(match_frac)

    slab_tag = getattr(slab, 'slab_id', getattr(slab, 'rank', -1))

    ax0, ax1 = axes[r, 0], axes[r, 1]
    im0 = ax0.imshow(cweb_slice.T, origin='lower', cmap=cmap, norm=norm, interpolation='nearest')
    ax0.set_title(f'CWEB from file (global ix={ixg}, slab={slab_tag}, match={match_frac:.3f})')
    ax0.set_xlabel('grid y')
    ax0.set_ylabel('grid z')
    # ax0.set_xlim(0, ngrid/5)
    # ax0.set_ylim(0, ngrid/5)

    im1 = ax1.imshow(cweb_from_eig.T, origin='lower', cmap=cmap, norm=norm, interpolation='nearest')
    ax1.set_title(f'CWEB from eig>= {SLICE_THRESHOLD:.2f} count')
    ax1.set_xlabel('grid y')
    ax1.set_ylabel('grid z')
    # ax1.set_xlim(0, ngrid/5)
    # ax1.set_ylim(0, ngrid/5)

legend_handles = [Patch(facecolor=class_colors[i], edgecolor='none', label=class_labels[i]) for i in range(4)]
fig.suptitle(f'2D Cosmic-Web Environment Slices (ngrid={ngrid}, box={boxsize:.0f} Mpc/h)', fontsize=15, y=1.1)
fig.legend(handles=legend_handles, loc='upper center', bbox_to_anchor=(0.5, 1.2), ncol=4, frameon=False)
fig.subplots_adjust(top=0.86)
out = OUT_DIR / 'tweb_cweb_2d_slices.png'
fig.savefig(out, dpi=180, bbox_inches='tight')
plt.show()
print(f'Saved: {out}')
print('Per-slice CWEB consistency (file vs eig-threshold-count):', [round(x, 4) for x in match_fracs])

# Optional: show eigenvalue fields for middle slice to inspect smoothness/structure.
ix_mid = ix_values[len(ix_values) // 2]
cweb_mid, eig_mid, slab_mid = _load_global_x_slice(primary_slabs, ix_mid)
fig, ax = plt.subplots(1, 3, figsize=(16, 4.8), constrained_layout=True)
for j, name in enumerate([r'$\lambda_1$', r'$\lambda_2$', r'$\lambda_3$']):
    im = ax[j].imshow(eig_mid[j].T, origin='lower', cmap='coolwarm', interpolation='nearest')
    ax[j].set_title(f'{name} slice (global ix={ix_mid})')
    ax[j].set_xlabel('grid y')
    ax[j].set_ylabel('grid z')
    # ax[j].set_xlim(0, ngrid/5)
    # ax[j].set_ylim(0, ngrid/5)
    cb = fig.colorbar(im, ax=ax[j])
    cb.set_label(name)
out2 = OUT_DIR / 'tweb_eigenvalue_2d_slice_mid.png'
fig.savefig(out2, dpi=180, bbox_inches='tight')
plt.show()
print(f'Saved: {out2}')

## 2D Density Slice Visualization

This section plots density-field slices corresponding to global x-slice indices.

- Input slabs: `abacus_z0200_density_slab_rank*.npz`
- Visualization: `log10(density + eps)`
- Works across grid sizes by selecting slices via global x-fractions.

In [ ]:
PATH2MOCKTWEB2= '/pscratch/sd/d/dkololgi/abacus/mocks_with_eigs/cutsky_BGS_z0.200_AbacusSummit_base_c000_ph000_with_tweb_eigs_ng2048_rs4_v2.fits'


In [ ]:
# Ensure overlay cell can resolve the mock catalog path even when run standalone.
CATALOG_OVERRIDE = '/pscratch/sd/d/dkololgi/abacus/mocks_with_eigs/cutsky_BGS_z0.200_AbacusSummit_base_c000_ph000_with_tweb_eigs_ng2048_rs4_v2.fits'
print('CATALOG_OVERRIDE =', CATALOG_OVERRIDE)

In [ ]:
# 2D density slices from slab outputs (global x-index aware)
import glob
from pathlib import Path

DENSITY_SLAB_DIR = Path('/pscratch/sd/d/dkololgi/AbscusSummit_densities')
DENSITY_PATTERN = 'abacus_z0200_density_slab_rank*.npz'
DENSITY_X_FRACTIONS = [0.2, 0.5, 0.8]
DENSITY_EPS = 1e-8


def _discover_density_slabs(slab_dir: Path, pattern: str):
    files = sorted(glob.glob(str(slab_dir / pattern)))
    if not files:
        raise FileNotFoundError(f'No density slab files found: {slab_dir / pattern}')
    items = []
    for p in files:
        with np.load(p) as d:
            items.append(
                {
                    'path': p,
                    'x_start': int(d['x_start']),
                    'x_end': int(d['x_end']),
                    'ngrid': int(d['ngrid']),
                    'boxsize': float(d['boxsize']),
                }
            )
    items.sort(key=lambda x: x['x_start'])
    return items


def _load_density_global_x_slice(items, global_ix: int):
    for it in items:
        if it['x_start'] <= global_ix < it['x_end']:
            lix = int(global_ix - it['x_start'])
            with np.load(it['path']) as d:
                dens = np.asarray(d['dens'][lix, :, :], dtype=np.float32)
            return dens, it
    raise ValueError(f'Global ix={global_ix} not covered by density slabs.')


density_items = _discover_density_slabs(DENSITY_SLAB_DIR, DENSITY_PATTERN)
ngrid_d = int(density_items[0]['ngrid'])
boxsize_d = float(density_items[0]['boxsize'])

# If T-Web slices are already configured, reuse their fractions by default.
x_fracs = DENSITY_X_FRACTIONS
if 'SLICE_X_FRACTIONS' in globals() and isinstance(SLICE_X_FRACTIONS, (list, tuple)) and len(SLICE_X_FRACTIONS) > 0:
    x_fracs = list(SLICE_X_FRACTIONS)

ix_vals = sorted({int(np.clip(round(f * (ngrid_d - 1)), 0, ngrid_d - 1)) for f in x_fracs})

print(f'Using density slabs from: {DENSITY_SLAB_DIR}')
print(f'Detected ngrid={ngrid_d}, boxsize={boxsize_d}, n_slabs={len(density_items)}')

fig, axes = plt.subplots(1, len(ix_vals), figsize=(5.5 * len(ix_vals), 4.8), constrained_layout=True)
if len(ix_vals) == 1:
    axes = [axes]

for ax, ixg in zip(axes, ix_vals):
    dens_slice, meta = _load_density_global_x_slice(density_items, ixg)
    img = np.log10(np.maximum(dens_slice, 0.0) + DENSITY_EPS)
    im = ax.imshow(img.T, origin='lower', cmap='magma', interpolation='nearest')
    slab_rank = Path(meta['path']).stem.split('rank')[-1]
    ax.set_title(f'log10(density+eps) at global ix={ixg}\nslab rank={slab_rank}')
    ax.set_xlabel('grid y')
    ax.set_ylabel('grid z')
    cb = fig.colorbar(im, ax=ax)
    cb.set_label('log10(density + eps)')
    ax.set_xlim(0, ngrid_d/5)
    ax.set_ylim(0, ngrid_d/5)

fig.suptitle(f'2D Density Field Slices (ngrid={ngrid_d}, box={boxsize_d:.0f} Mpc/h)', y=1.02)
out = OUT_DIR / 'density_2d_slices.png'
fig.savefig(out, dpi=250, bbox_inches='tight')
plt.show()
print(f'Saved: {out}')

## Galaxy Overlay on T-Web Slice

Overlay mock galaxies (as `x` markers) on a 2D CWEB slice to visually inspect whether galaxies trace the cosmic-web structure.

- Uses `PRIMARY_DIR` slab outputs.
- Uses `CATALOG_OVERRIDE` if set (otherwise resolves from metadata).
- Position source priority:
  1. direct box-frame columns (`X/Y/Z` or `x/y/z`) in the mock catalog,
  2. fallback via `(FILE_NUM, HALO_INDEX)` -> `halo_info` host-halo positions (`x_com` by default).

In [ ]:
PATH2MOCKTWEB2= '/pscratch/sd/d/dkololgi/abacus/mocks_with_eigs/cutsky_BGS_z0.200_AbacusSummit_base_c000_ph000_with_tweb_eigs_ng2048_rs4_v2.fits'
data = Table(fitsio.read(PATH2MOCKTWEB))
data[(data['IN_Y1']==1) & (data['IN_Y5'])==1]

In [ ]:
# Overlay mock galaxies on a selected T-Web CWEB slice
from pathlib import Path as _Path
import numpy as _np
import fitsio as _fitsio

OVERLAY_X_FRAC = 0.5           # global x-slice fraction in [0,1]
OVERLAY_X_HALF_WIDTH_CELLS = 1  # include galaxies within +/- this many x-cells
OVERLAY_MAX_POINTS = 20000       # cap points for readability/performance
OVERLAY_MARKER_SIZE = 1         # bigger markers for visibility
OVERLAY_ALPHA = 0.95
OVERLAY_MARKER = 'o'             # 'o' is easier to see than 'x' on dense backgrounds
OVERLAY_FORCE_POINT_COLOR = '#ffffff'
OVERLAY_EDGE_COLOR = '#000000'
OVERLAY_EDGE_WIDTH = 0.35
OVERLAY_AUTO_ZOOM = True         # zoom to selected points for visual sanity check
OVERLAY_ZOOM_MARGIN_CELLS = 24
OVERLAY_CATALOG_PATH = '/pscratch/sd/d/dkololgi/abacus/mocks_with_eigs/cutsky_BGS_z0.200_AbacusSummit_base_c000_ph000_with_tweb_eigs_ng2048_rs4_v2.fits'
HALO_POS_FIELD = 'x_com'        # fallback host-halo position field: x_com or x_L2com
HALO_INFO_DIR = _Path('/global/cfs/cdirs/desi/public/cosmosim/AbacusSummit/AbacusSummit_base_c000_ph000/halos/z0.200/halo_info')


def _resolve_col_any(names, candidates):
    m = {n.upper(): n for n in names}
    for c in candidates:
        if c.upper() in m:
            return m[c.upper()]
    return None


def _catalog_path_for_overlay():
    # Prefer explicit local override so this cell runs standalone.
    if 'OVERLAY_CATALOG_PATH' in globals() and OVERLAY_CATALOG_PATH:
        p = _Path(OVERLAY_CATALOG_PATH).expanduser().resolve()
        if p.exists():
            return p

    # Then use notebook-level override if present.
    if 'CATALOG_OVERRIDE' in globals() and CATALOG_OVERRIDE:
        p = _Path(CATALOG_OVERRIDE).expanduser().resolve()
        if p.exists():
            return p

    # Then any previously resolved source_catalog variable.
    if 'source_catalog' in globals() and source_catalog is not None:
        p = _Path(source_catalog).expanduser().resolve()
        if p.exists():
            return p

    # Finally metadata resolver, if defined.
    if '_resolve_source_catalog_path' in globals() and 'gnn_meta' in globals():
        p = _Path(_resolve_source_catalog_path(gnn_meta, None)).expanduser().resolve()
        if p.exists():
            return p

    raise RuntimeError(
        'Could not resolve mock catalog path. Set OVERLAY_CATALOG_PATH in this cell or CATALOG_OVERRIDE in config.'
    )


def _load_mock_box_xyz(catalog_path: _Path, *, apply_y1y5_filter: bool = False):
    """Load box-frame xyz. If apply_y1y5_filter, keep IN_Y1==1 or IN_Y5==1 (same as export_cutsky_boxframe_points)."""

    def _y1y5_mask(table: _np.ndarray) -> _np.ndarray:
        u = {n.upper(): n for n in table.dtype.names}
        a1 = u.get('IN_Y1')
        a5 = u.get('IN_Y5')
        if a1 is None and a5 is None:
            return _np.ones(len(table), dtype=bool)
        m = _np.zeros(len(table), dtype=bool)
        if a1 is not None:
            m |= table[a1] == 1
        if a5 is not None:
            m |= table[a5] == 1
        if not m.any():
            m[:] = True
        return m

    # Try direct box-frame columns first.
    base_cols = ['FILE_NUM', 'HALO_INDEX', 'CWEB', 'LAMBDA1', 'LAMBDA2', 'LAMBDA3']
    with _fitsio.FITS(str(catalog_path)) as f:
        names = list(f[1].get_colnames())

    cx = _resolve_col_any(names, ('X', 'x'))
    cy = _resolve_col_any(names, ('Y', 'y'))
    cz = _resolve_col_any(names, ('Z', 'z', 'Z_COSMO'))

    if cx and cy and cz:
        cols = [cx, cy, cz] + [c for c in base_cols if c in names]
        um = {n.upper(): n for n in names}
        if apply_y1y5_filter:
            for k in ('IN_Y1', 'IN_Y5'):
                if k in um and um[k] not in cols:
                    cols.append(um[k])
        tab = _fitsio.read(str(catalog_path), columns=cols)
        if apply_y1y5_filter:
            tab = tab[_y1y5_mask(tab)]
        xyz = _np.stack([
            _np.asarray(tab[cx], dtype=_np.float64),
            _np.asarray(tab[cy], dtype=_np.float64),
            _np.asarray(tab[cz], dtype=_np.float64),
        ], axis=1)
        return tab, xyz, 'direct'

    # Fallback: build xyz from (FILE_NUM, HALO_INDEX) using halo_info host positions.
    fn_col = _resolve_col_any(names, ('FILE_NUM',))
    hi_col = _resolve_col_any(names, ('HALO_INDEX',))
    if not fn_col or not hi_col:
        raise RuntimeError('Catalog has no direct xyz columns and no FILE_NUM/HALO_INDEX linkage.')

    cols = [fn_col, hi_col] + [c for c in base_cols if c in names and c not in (fn_col, hi_col)]
    um = {n.upper(): n for n in names}
    if apply_y1y5_filter:
        for k in ('IN_Y1', 'IN_Y5'):
            if k in um and um[k] not in cols:
                cols.append(um[k])
    tab = _fitsio.read(str(catalog_path), columns=cols)
    if apply_y1y5_filter:
        tab = tab[_y1y5_mask(tab)]

    file_num = _np.asarray(tab[fn_col], dtype=_np.int16)
    halo_idx = _np.asarray(tab[hi_col], dtype=_np.int32)
    xyz = _np.full((len(tab), 3), _np.nan, dtype=_np.float64)

    try:
        from abacusnbody.data.compaso_halo_catalog import CompaSOHaloCatalog as _CompaSOHaloCatalog
    except Exception as e:
        raise RuntimeError('abacusnbody is required for halo-link fallback.') from e

    uniq_fn = _np.unique(file_num[file_num >= 0])
    for fn in uniq_fn:
        sel = _np.where(file_num == fn)[0]
        hp = HALO_INFO_DIR / f'halo_info_{int(fn):03d}.asdf'
        if not hp.exists():
            continue
        # Handle API differences across abacusnbody versions.
        try:
            cat = _CompaSOHaloCatalog(
                str(hp),
                fields=[HALO_POS_FIELD],
                cleaned=True,
                convert_units=True,
                unpack_bits=False,
                verbose=False,
            )
        except (TypeError, ValueError):
            # Older/newer signatures may not accept some kwargs.
            cat = _CompaSOHaloCatalog(str(hp), cleaned=True)

        arr = _np.asarray(cat.halos[HALO_POS_FIELD], dtype=_np.float64)
        idx = halo_idx[sel]
        valid = (idx >= 0) & (idx < arr.shape[0])
        if _np.any(valid):
            xyz[sel[valid]] = arr[idx[valid]]

    good = _np.all(_np.isfinite(xyz), axis=1)
    if not _np.any(good):
        raise RuntimeError('Failed to recover any finite box-frame xyz from halo linkage.')

    if (~good).any():
        tab = tab[good]
        xyz = xyz[good]

    return tab, xyz, 'halo_link'



# Reload slabs from PRIMARY_DIR to avoid stale state.
primary_slabs = discover_slabs(PRIMARY_DIR)
ngrid = int(primary_slabs[0].ngrid)
boxsize = float(primary_slabs[0].boxsize)
cell = boxsize / ngrid
ixg = int(_np.clip(round(OVERLAY_X_FRAC * (ngrid - 1)), 0, ngrid - 1))

# Load CWEB slice and eig slice at selected global x.
cweb_slice, eig_slice, slab = _load_global_x_slice(primary_slabs, ixg)

# Load mock and determine xyz positions.
catalog_path = _catalog_path_for_overlay()
mock_tab, xyz, xyz_source = _load_mock_box_xyz(catalog_path, apply_y1y5_filter=False)

# Convert xyz to voxel indices.
xyz_mod = _np.mod(xyz, boxsize)
ix = _np.floor(xyz_mod[:, 0] / cell).astype(_np.int32)
iy = _np.floor(xyz_mod[:, 1] / cell).astype(_np.int32)
iz = _np.floor(xyz_mod[:, 2] / cell).astype(_np.int32)

# Select galaxies near the chosen x-slice.
sel = _np.abs(ix - ixg) <= int(OVERLAY_X_HALF_WIDTH_CELLS)
idx = _np.where(sel)[0]

if idx.size == 0:
    raise RuntimeError('No galaxies selected for requested x-slice and width.')

n_selected_full = int(idx.size)
if idx.size > OVERLAY_MAX_POINTS:
    rng = _np.random.default_rng(123)
    idx = rng.choice(idx, size=OVERLAY_MAX_POINTS, replace=False)

print(f'Galaxies in slab selection: {n_selected_full:,}; plotted: {idx.size:,}')

# Marker color: force high-contrast color by default for visibility.
point_colors = OVERLAY_FORCE_POINT_COLOR

# Plot base CWEB map + galaxy overlay as x markers.
from matplotlib.colors import ListedColormap as _ListedColormap, BoundaryNorm as _BoundaryNorm
from matplotlib.patches import Patch as _Patch

class_colors = ['#1f77b4', '#2ca02c', '#ff7f0e', '#d62728']
class_labels = ['0: void', '1: sheet/wall', '2: filament', '3: knot']
cmap = _ListedColormap(class_colors)
norm = _BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5], cmap.N)

fig, ax = plt.subplots(1, 1, figsize=(8.2, 7.2), constrained_layout=True)
im = ax.imshow(cweb_slice.T, origin='lower', cmap=cmap, norm=norm, interpolation='nearest')

ax.scatter(
    iy[idx],
    iz[idx],
    marker=OVERLAY_MARKER,
    s=OVERLAY_MARKER_SIZE,
    c=point_colors,
    alpha=OVERLAY_ALPHA,
    edgecolors=OVERLAY_EDGE_COLOR,
    linewidths=OVERLAY_EDGE_WIDTH,
    zorder=3,
)

ax.set_title(
    f'Mock Galaxies over CWEB Slice (global ix={ixg}, width=+/-{OVERLAY_X_HALF_WIDTH_CELLS} cells)\n'
    f'source={catalog_path.name}, xyz={xyz_source}, n_points={idx.size:,}'
)
ax.set_xlabel('grid y')
ax.set_ylabel('grid z')

if OVERLAY_AUTO_ZOOM and idx.size > 0:
    y0, y1 = int(_np.min(iy[idx])), int(_np.max(iy[idx]))
    z0, z1 = int(_np.min(iz[idx])), int(_np.max(iz[idx]))
    m = int(max(0, OVERLAY_ZOOM_MARGIN_CELLS))
    ax.set_xlim(max(0, y0 - m), min(ngrid - 1, y1 + m))
    ax.set_ylim(max(0, z0 - m), min(ngrid - 1, z1 + m))

legend_handles = [_Patch(facecolor=class_colors[i], edgecolor='none', label=class_labels[i]) for i in range(4)]
leg = ax.legend(handles=legend_handles, loc='upper right', frameon=True, title='CWEB')
leg.get_frame().set_alpha(0.8)

cb = fig.colorbar(im, ax=ax, ticks=[0, 1, 2, 3])
cb.set_label('CWEB class')

out = OUT_DIR / 'tweb_mock_overlay_cweb_slice.png'
# fig.savefig(out, dpi=220, bbox_inches='tight')
plt.show()
print(f'Saved: {out}')
print(f'PRIMARY_DIR={PRIMARY_DIR}')
print(f'Catalog={catalog_path}')
print(f'xyz source={xyz_source}')

In [ ]:
# YZ grid scatter (no CWEB): random mock subsample over the *entire* periodic box (all ix slabs).
# Same voxel mapping as overlay: mod(xyz,L), floor(.../cell) -> (ix,iy,iz). Plot iy vs iz (Y-Z projection).
# Run the overlay cell above first: OVERLAY_* , _catalog_path_for_overlay, _load_mock_box_xyz, plt, OUT_DIR, etc.
import numpy as _np
import matplotlib.pyplot as _plt

WHOLE_CUBE_SUBSAMPLE = 700_000
WHOLE_CUBE_SEED = 123
# If True, keep only rows with IN_Y1==1 or IN_Y5==1 (same convention as export_cutsky_boxframe_points).
WHOLE_CUBE_Y1Y5_ONLY = True
# 'full' = [0, ngrid-1] on both axes (entire YZ footprint); 'zoom' = tight around subsample (+ margin)
FULL_CUBE_YZ_GRID_LIM = 'full'
_ZOOM_MARGIN = int(globals().get('OVERLAY_ZOOM_MARGIN_CELLS', 24))

if '_catalog_path_for_overlay' not in globals() or '_load_mock_box_xyz' not in globals():
    raise RuntimeError('Run the overlay cell above first (defines overlay helpers and OVERLAY_*).')
if 'discover_slabs' not in globals() or 'PRIMARY_DIR' not in globals():
    raise RuntimeError('discover_slabs and PRIMARY_DIR must be defined (earlier cells).')

primary_slabs = discover_slabs(PRIMARY_DIR)
ngrid = int(primary_slabs[0].ngrid)
boxsize = float(primary_slabs[0].boxsize)
cell = boxsize / ngrid

catalog_path = _catalog_path_for_overlay()
_mock_only, xyz, xyz_source = _load_mock_box_xyz(catalog_path, apply_y1y5_filter=bool(WHOLE_CUBE_Y1Y5_ONLY))

n_tot = int(len(xyz))
xyz_mod = _np.mod(xyz, boxsize)
iy = _np.floor(xyz_mod[:, 1] / cell).astype(_np.int32)
iz = _np.floor(xyz_mod[:, 2] / cell).astype(_np.int32)

_rng = _np.random.default_rng(int(WHOLE_CUBE_SEED))
_n_draw = min(int(WHOLE_CUBE_SUBSAMPLE), n_tot)
idx = _rng.choice(n_tot, size=_n_draw, replace=False)

fig, ax = _plt.subplots(1, 1, figsize=(8.2, 7.2), constrained_layout=True)
ax.scatter(
    iy[idx],
    iz[idx],
    marker=OVERLAY_MARKER,
    s=0.5,
    c=OVERLAY_FORCE_POINT_COLOR,
    alpha=OVERLAY_ALPHA,
    edgecolors=OVERLAY_EDGE_COLOR,
    linewidths=0.0,
    zorder=3,
    rasterized=True,
)
ax.set_xlabel('grid y')
ax.set_ylabel('grid z')
ax.set_aspect('equal')
_y1tag = 'IN_Y1|IN_Y5 only | ' if WHOLE_CUBE_Y1Y5_ONLY else ''
ax.set_title(
    f'Random subsample ({_n_draw:,} / {n_tot:,}) — YZ grid (all x slabs), no CWEB\n'
    f'{_y1tag}source={catalog_path.name}, xyz={xyz_source}'
)

if str(FULL_CUBE_YZ_GRID_LIM).lower() == 'full':
    ax.set_xlim(0, ngrid - 1)
    ax.set_ylim(0, ngrid - 1)
elif str(FULL_CUBE_YZ_GRID_LIM).lower() == 'zoom':
    y0, y1 = int(_np.min(iy[idx])), int(_np.max(iy[idx]))
    z0, z1 = int(_np.min(iz[idx])), int(_np.max(iz[idx]))
    m = int(max(0, _ZOOM_MARGIN))
    ax.set_xlim(max(0, y0 - m), min(ngrid - 1, y1 + m))
    ax.set_ylim(max(0, z0 - m), min(ngrid - 1, z1 + m))

_out_only = OUT_DIR / 'tweb_mock_yz_grid_full_box_subsample_no_cweb.png'
fig.savefig(_out_only, dpi=220, bbox_inches='tight')
_plt.show()
print(f'Saved: {_out_only}')
print(
    f'  ngrid={ngrid} L={boxsize:.1f} Mpc | drawn={_n_draw:,} / {n_tot:,} | '
    f'xyz_source={xyz_source} | Y1Y5_only={WHOLE_CUBE_Y1Y5_ONLY} | axis={FULL_CUBE_YZ_GRID_LIM}'
)


In [ ]:
# Pearson r + MI: same x-slab as overlay; CWEB from T-Web slab NPZs at (ix,iy,iz) — NOT from FITS.
# PRIMARY_DIR must match the T-Web run you trust for the overlay. Optional: report FITS CWEB agreement.
# Requires: overlay cell (_catalog_path_for_overlay, _load_mock_box_xyz), discover_slabs, PRIMARY_DIR.
import numpy as _np
from scipy.stats import pearsonr as _pearsonr

WHOLE_CUBE_SUBSAMPLE = 200_000
WHOLE_CUBE_SEED = 123
MI_RANDOM_STATE = 0
MI_N_NEIGHBORS = 5

_SLAB_FRAC = float(globals().get('OVERLAY_X_FRAC', 0.5))
_SLAB_HW = int(globals().get('OVERLAY_X_HALF_WIDTH_CELLS', 1))

if '_catalog_path_for_overlay' not in globals() or '_load_mock_box_xyz' not in globals():
    raise RuntimeError('Run the overlay cell first.')
if 'discover_slabs' not in globals() or 'PRIMARY_DIR' not in globals():
    raise RuntimeError('discover_slabs / PRIMARY_DIR required.')

try:
    from sklearn.feature_selection import mutual_info_classif as _mi_classif
except ImportError:
    _mi_classif = None
    print('sklearn not available; MI will be skipped. Install scikit-learn for MI.')


def _validate_and_build_slab_maps(_slabs: list):
    _ngrid_set = {int(s.ngrid) for s in _slabs}
    _box_set = {float(s.boxsize) for s in _slabs}
    if len(_ngrid_set) != 1 or len(_box_set) != 1:
        raise ValueError('Inconsistent ngrid/boxsize across slab files.')
    _ngrid = int(next(iter(_ngrid_set)))
    _ix_to_slab = _np.full(_ngrid, -1, dtype=_np.int16)
    _slab_xstart = _np.full(len(_slabs), -1, dtype=_np.int32)
    _expected = 0
    for _si, _s in enumerate(_slabs):
        if int(_s.x_start) != _expected:
            raise ValueError(f'Slab gap/overlap at x={_expected}, next starts {_s.x_start}')
        if int(_s.x_end) <= int(_s.x_start):
            raise ValueError(f'Invalid slab [{_s.x_start}, {_s.x_end})')
        _ix_to_slab[int(_s.x_start) : int(_s.x_end)] = _si
        _slab_xstart[_si] = int(_s.x_start)
        _expected = int(_s.x_end)
    if _expected != _ngrid or _np.any(_ix_to_slab < 0):
        raise ValueError('Slab x-coverage incomplete vs ngrid.')
    return _ix_to_slab, _slab_xstart, _ngrid


def _gather_cweb_at_cells(_slabs: list, _ix_to_slab: _np.ndarray, _slab_xstart: _np.ndarray, _ix, _iy, _iz):
    _ix = _np.asarray(_ix, dtype=_np.int64).ravel()
    _iy = _np.asarray(_iy, dtype=_np.int64).ravel()
    _iz = _np.asarray(_iz, dtype=_np.int64).ravel()
    _n = int(_ix.size)
    _out = _np.empty(_n, dtype=_np.uint8)
    _sid = _ix_to_slab[_ix]
    if _np.any(_sid < 0):
        raise ValueError('Some ix are outside [0, ngrid); check voxel indices.')
    _local_ix = _ix - _slab_xstart[_sid.astype(_np.int64)]
    for _si, _s in enumerate(_slabs):
        _rows = _np.nonzero(_sid == _si)[0]
        if _rows.size == 0:
            continue
        with _np.load(_s.path) as _d:
            _c_loc = _d['cweb']
        _li = _local_ix[_rows].astype(_np.int64)
        _yj = _iy[_rows].astype(_np.int64)
        _zk = _iz[_rows].astype(_np.int64)
        _out[_rows] = _c_loc[_li, _yj, _zk]
    return _out


_slabs = discover_slabs(PRIMARY_DIR)
_ix_to_slab, _slab_xstart, _ngrid = _validate_and_build_slab_maps(_slabs)
_L = float(_slabs[0].boxsize)
_cell = _L / _ngrid
_ixg = int(_np.clip(round(_SLAB_FRAC * (_ngrid - 1)), 0, _ngrid - 1))

_cat = _catalog_path_for_overlay()
_tab, _xyz, _src = _load_mock_box_xyz(_cat)
_names = {n.upper(): n for n in _tab.dtype.names}
_cweb_fits_col = _names.get('CWEB')

_n = int(len(_tab))
_xyz_m = _np.mod(_np.asarray(_xyz, dtype=_np.float64), _L)
_ix = _np.floor(_xyz_m[:, 0] / _cell).astype(_np.int32)
_iy = _np.floor(_xyz_m[:, 1] / _cell).astype(_np.int32)
_iz = _np.floor(_xyz_m[:, 2] / _cell).astype(_np.int32)
_np.clip(_ix, 0, _ngrid - 1, out=_ix)
_np.clip(_iy, 0, _ngrid - 1, out=_iy)
_np.clip(_iz, 0, _ngrid - 1, out=_iz)

_slab_mask = _np.abs(_ix.astype(_np.int64) - int(_ixg)) <= int(_SLAB_HW)
_idx_slab = _np.flatnonzero(_slab_mask)
_n_slab = int(_idx_slab.size)
if _n_slab == 0:
    raise RuntimeError(f'No galaxies in x-slab ixg={_ixg} ±{_SLAB_HW}')

_rng = _np.random.default_rng(int(WHOLE_CUBE_SEED))
_n_draw = min(int(WHOLE_CUBE_SUBSAMPLE), _n_slab)
_sub = _rng.choice(_idx_slab, size=_n_draw, replace=False)

_ixs = _ix[_sub].astype(_np.int64)
_iys = _iy[_sub].astype(_np.int64)
_izs = _iz[_sub].astype(_np.int64)
_xyz_s = _xyz_m[_sub]

_cweb_np = _gather_cweb_at_cells(_slabs, _ix_to_slab, _slab_xstart, _ixs, _iys, _izs).astype(_np.int64)

_msk = (_cweb_np >= 0) & (_cweb_np <= 3)
if not _np.all(_msk):
    _cweb = _cweb_np[_msk]
    _xyz_s = _xyz_s[_msk]
    _ixs = _ixs[_msk]
    _iys = _iys[_msk]
    _izs = _izs[_msk]
    print(f'  (dropped {int(_np.size(_msk) - _cweb.size)} rows with CWEB_grid not in 0..3)')
else:
    _cweb = _cweb_np

_cweb_f = _cweb.astype(_np.float64)

print(
    f'PRIMARY_DIR={PRIMARY_DIR}\n'
    f'catalog={_cat.name} | xyz_source={_src} | stats use CWEB from slab NPZ at (ix,iy,iz), not FITS\n'
    f'slab: ixg={_ixg} ±{_SLAB_HW} cells | n_slab={_n_slab:,} | subsample={_cweb.size:,}'
)
if _cweb_fits_col is not None:
    _fits_np = _np.asarray(_tab[_cweb_fits_col][_sub], dtype=_np.int64).ravel()
    _cmp = (_cweb_np >= 0) & (_cweb_np <= 3) & (_fits_np >= 0) & (_fits_np <= 3)
    if _np.any(_cmp):
        _ag = float(_np.mean(_fits_np[_cmp] == _cweb_np[_cmp]))
        print(
            f'  Sanity: FITS CWEB vs grid lookup agreement on same rows: {_ag:.4f}  '
            f'(low ⇒ FITS annotation used a different T-Web run than PRIMARY_DIR)'
        )
print(f'  ix range: [{int(_ixs.min())}, {int(_ixs.max())}]')
print('--- Pearson r(coord, CWEB_grid) [exploratory; CWEB coded 0–3] ---')
for _lab, _col in zip(('ix', 'iy', 'iz'), (_ixs.astype(_np.float64), _iys.astype(_np.float64), _izs.astype(_np.float64))):
    _r, _p = _pearsonr(_col, _cweb_f)
    print(f'  {_lab} (grid):  r={_r:+.5f}  p={_p:.3e}')
for _lab, _j in zip(('x', 'y', 'z'), (0, 1, 2)):
    _r, _p = _pearsonr(_xyz_s[:, _j], _cweb_f)
    print(f'  {_lab} (Mpc mod L):  r={_r:+.5f}  p={_p:.3e}')

_Y2 = _np.column_stack((_iys.astype(_np.float64), _izs.astype(_np.float64)))
_X3 = _np.column_stack((_ixs.astype(_np.float64), _iys.astype(_np.float64), _izs.astype(_np.float64)))

if _mi_classif is not None:
    print('--- Mutual information (nats, sklearn) vs CWEB_grid ---')
    _mi_yz = _mi_classif(
        _Y2,
        _cweb,
        discrete_features=True,
        n_neighbors=int(MI_N_NEIGHBORS),
        random_state=int(MI_RANDOM_STATE),
    )
    _mi_xyz = _mi_classif(
        _X3,
        _cweb,
        discrete_features=True,
        n_neighbors=int(MI_N_NEIGHBORS),
        random_state=int(MI_RANDOM_STATE),
    )
    _mi_mpc = _mi_classif(
        _xyz_s,
        _cweb,
        discrete_features=False,
        n_neighbors=int(MI_N_NEIGHBORS),
        random_state=int(MI_RANDOM_STATE),
    )
    print(f'  MI vs CWEB per axis [iy, iz] (grid): {_mi_yz}')
    print(f'  MI vs CWEB per axis [ix, iy, iz] (grid): {_mi_xyz}')
    print(f'  MI vs CWEB per axis [x, y, z] (Mpc mod L): {_mi_mpc}')
    print(f'  (n_neighbors={MI_N_NEIGHBORS})')


### Box-frame graph edges in the same slice (projection)

The CWEB overlay cell indexes **FITS rows** (full catalog). The GNN graph uses the **Y1/Y5-filtered** row order saved in `abacus_alpha_boxframe_points.npy` / `*_node_features.parquet`. This cell loads **graph artifacts** so **node `i` = row `i`** in the parquet, finds edges whose **both** endpoints lie in the same `ix` slab as the overlay, and plots them in **(grid y, grid z)** in a **second figure** (no T-Web background) to sanity-check local connectivity.

**Important:** The default **YZ LineCollection is not a 2D slice of the graph** — it **projects every 3D edge onto the YZ plane and ignores Δx**. Delaunay edges often span several x-cells inside that thick slab, so they pile up in projection and can look like **false “slabs”, ribbons, or over-connected layers** even when the **3D** triangulation is correct. The notebook now adds (1) **diagnostics** (median |Δix|, 3D edge length), (2) a **same-ix edge** panel (only edges whose endpoints share the same `ix` cell), and (3) a **3D** edge view in Mpc.

**Illustris-style slab figure (first plot in the code cell):** ~**20 Mpc** thick slab through **box center**, **comoving Mpc** on the two tangential axes (default **Z**-normal → **X vs Y**), and **only edges whose both endpoints lie inside the slab** (same rule as “don’t draw edges that leave the slice”). That removes **integer grid snapping**, which was producing lots of **axis-aligned** segments in the T-Web–matched **(iy, iz)** figures. *Caveat:* some TNG papers build a **2D Delaunay triangulation in the slice**; this notebook keeps your **precomputed 3D alpha/Delaunay** edges and merely **filters** them to the slab — topology can differ slightly from a fresh 2D DT on the same points.

**Why a CutSky “boxframe” slab can look like a lightcone:** The halo positions are still simulation **comoving box** coordinates, but the **galaxy sample** is **DESI-style CutSky** (Y1/Y5), i.e. **survey / lightcone selection** — not a volume-limited fair cube like IllustrisTNG300. Compare to `ABACUS_TWEB_AUDIT_FINDINGS.md`: folding with `% boxsize` for T-Web can also distort geometry. The slab QC plot defaults to **`ILLUSTRIS_COORDS_FOR_SLAB='native'`** (same as `points_xyz.npy` / Gudhi); use `'periodic'` only if you explicitly want `np.mod(box)` like the voxel map.

**Halo-built graph vs sky Cartesian (cell 28 `sky_cartesian` figure):** The **edge list** comes from **Gudhi on halo `x_com`**. Plotting those same edges with vertices at **RA/Dec/Z → Cartesian (Planck18)** draws segments between nodes that are **neighbors in halo space** but often **far apart in sky space** — long chords and “bridges” are **expected**, not a bug in Delaunay. That **frame mismatch** (graph metrics / degree / clustering defined in one embedding, labels or physics summarized in another) is a **prime reason** for **weak feature–target correlation** (e.g. vs T-Web eigenvalues). Remediation: **one consistent frame end-to-end** — either rebuild the graph in sky Cartesian to match observables, or keep halo-frame graphs and assign/compare labels using **halo-consistent** voxel mapping (see audit doc on `(FILE_NUM, HALO_INDEX)` and avoiding naive `% box` on sky coords).



In [ ]:
# Efficient edge plot: show up to 1M galaxy edges in interactive Plotly, XY projection (Mpc)
import numpy as np
import plotly.graph_objects as go
from pathlib import Path

GRAPH_QC_DIR = Path('/pscratch/sd/d/dkololgi/abacus/graph_constructions')
GRAPH_QC_PREFIX = 'abacus_mock_alpha_23032026'
N_EDGES_DRAW = 1_000_000  # Plot up to 1 million edges

pts_path = GRAPH_QC_DIR / f'{GRAPH_QC_PREFIX}_points_xyz.npy'
edg_path = GRAPH_QC_DIR / f'{GRAPH_QC_PREFIX}_edges_combined_idx.npy'
if not pts_path.exists() or not edg_path.exists():
    raise FileNotFoundError(f'Missing {pts_path} or {edg_path}')

xyz = np.load(pts_path).astype(np.float64)
edges = np.load(edg_path).astype(np.int64)

if edges.ndim != 2 or edges.shape[1] != 2:
    raise ValueError(f'Unexpected edges shape {edges.shape}')
if edges.size and int(edges.max()) >= len(xyz):
    raise ValueError(f'Edge index out of range: max={int(edges.max())}, n_pts={len(xyz)}')

n_edges = min(N_EDGES_DRAW, edges.shape[0])
rng = np.random.default_rng(20260324)
sel = rng.choice(edges.shape[0], size=n_edges, replace=False)
sampled_edges = edges[sel]
i0, i1 = sampled_edges[:, 0], sampled_edges[:, 1]

# Make a single large scattergl trace for all segments to optimize performance
x0 = xyz[i0, 0]
y0 = xyz[i0, 1]
x1 = xyz[i1, 0]
y1 = xyz[i1, 1]

# Interleave start/end of each segment. Use None for gap between segments.
# This allows Plotly to treat each [x0, x1] as a separate line
plot_x = np.empty(3 * n_edges)
plot_y = np.empty(3 * n_edges)
plot_x[::3], plot_y[::3] = x0, y0
plot_x[1::3], plot_y[1::3] = x1, y1
plot_x[2::3], plot_y[2::3] = np.nan, np.nan  # for line breaks

margin = 0.02 * np.max([np.ptp(xyz[:,0]), np.ptp(xyz[:,1])])
xlims = [xyz[:,0].min() - margin, xyz[:,0].max() + margin]
ylims = [xyz[:,1].min() - margin, xyz[:,1].max() + margin]

fig = go.Figure()
fig.add_trace(go.Scattergl(
    x=plot_x, y=plot_y,
    mode='lines',
    line=dict(color='#2ec4ff', width=1),
    opacity=0.35,
    hoverinfo='skip',
    name=f'Edges ({n_edges:,})',
))
fig.update_layout(
    title=f'Graph QC (2D xy): {n_edges:,} edges (cyan lines projected onto xy plane)<br>{GRAPH_QC_PREFIX}',
    xaxis=dict(title='x [Mpc]', range=[-1500, 1500], showgrid=False, zeroline=False),
    yaxis=dict(title='y [Mpc]', range=[-1500, 1500], showgrid=False, zeroline=False, scaleanchor='x', scaleratio=1),
    plot_bgcolor='#0a0a0c',
    paper_bgcolor='#0a0a0c',
    font=dict(color='white'),
    margin=dict(l=0, r=0, t=50, b=0),
    showlegend=False,
    width=1000, height=750
)
fig.show()

In [ ]:
# 200 Mpc cube: galaxies + edges with both endpoints inside (comoving Mpc, Plotly 3D)
#
# Coordinates are observer-frame Cartesian from RA/Dec/Z (build_abacus_graph), NOT a periodic
# simulation box. The mock is anisotropic — a cube centered e.g. at (-500,-500,-500) can contain
# **zero** galaxies. Use CUBE_MODE='median' (default) to pick a dense region, or set CUBE_MODE='manual'
# after inspecting the printed min/max/median below.

import numpy as np
import plotly.graph_objects as go
from pathlib import Path

GRAPH_QC_DIR = Path('/pscratch/sd/d/dkololgi/abacus/graph_constructions')
GRAPH_QC_PREFIX = 'abacus_mock_alpha_23032026'

# 'median' = center on median(x,y,z) of full catalog (good default). 'manual' = use CUBE_CENTER_MPC.
CUBE_MODE = 'median'
CUBE_CENTER_MPC = (-300.0, -500.0, -300.0)  # only if CUBE_MODE == 'manual'
CUBE_SIDE_MPC = 600.0  # [center - side/2, center + side/2] on each axis

_pts = GRAPH_QC_DIR / f'{GRAPH_QC_PREFIX}_points_xyz.npy'
_edg = GRAPH_QC_DIR / f'{GRAPH_QC_PREFIX}_edges_combined_idx.npy'
if not _pts.exists() or not _edg.exists():
    raise FileNotFoundError(f'Missing {_pts} or {_edg}')

_xyz = np.load(_pts).astype(np.float64)
_edges = np.load(_edg).astype(np.int64)
if _edges.ndim != 2 or _edges.shape[1] != 2:
    raise ValueError(f'Unexpected edges shape {_edges.shape}')
if _edges.size and int(_edges.max()) >= len(_xyz):
    raise ValueError(f'Edge index OOB: max={int(_edges.max())}, n_pts={len(_xyz)}')

print(f'{GRAPH_QC_PREFIX}: N={len(_xyz):,} | full-catalog axis ranges (Mpc):')
for _i, _lab in enumerate('xyz'):
    _col = _xyz[:, _i]
    print(
        f'  {_lab}: min={_col.min():.2f}  max={_col.max():.2f}  median={float(np.median(_col)):.2f}'
    )

_half = 0.5 * float(CUBE_SIDE_MPC)
if str(CUBE_MODE).lower() == 'median':
    _c = np.median(_xyz, axis=0)
    print(f"CUBE_MODE=median → center = ({_c[0]:.2f}, {_c[1]:.2f}, {_c[2]:.2f})")
elif str(CUBE_MODE).lower() == 'manual':
    _c = np.asarray(CUBE_CENTER_MPC, dtype=np.float64)
    print(f"CUBE_MODE=manual → center = ({_c[0]:.2f}, {_c[1]:.2f}, {_c[2]:.2f})")
else:
    raise ValueError("CUBE_MODE must be 'median' or 'manual'")

_lo, _hi = _c - _half, _c + _half

_in = (
    (_xyz[:, 0] >= _lo[0])
    & (_xyz[:, 0] <= _hi[0])
    & (_xyz[:, 1] >= _lo[1])
    & (_xyz[:, 1] <= _hi[1])
    & (_xyz[:, 2] >= _lo[2])
    & (_xyz[:, 2] <= _hi[2])
)
_n_gal = int(_in.sum())
_idx = np.flatnonzero(_in)

_em = _in[_edges[:, 0]] & _in[_edges[:, 1]]
_sub = _edges[_em]
_n_ed = int(_sub.shape[0])
_i0, _i1 = _sub[:, 0], _sub[:, 1]

if _n_gal == 0:
    print(
        '\n*** No galaxies in this cube. Observer-frame coords are not uniform in a box; '
        'try CUBE_MODE="median" or move CUBE_CENTER_MPC into the printed min/max ranges.\n'
    )

print(
    f'cube side={CUBE_SIDE_MPC:g} Mpc | bounds x[{_lo[0]:.1f},{_hi[0]:.1f}] '
    f'y[{_lo[1]:.1f},{_hi[1]:.1f}] z[{_lo[2]:.1f},{_hi[2]:.1f}]\n'
    f'  galaxies in cube: {_n_gal:,} | edges (both ends in cube): {_n_ed:,}'
)

if _n_ed > 0:
    _x = np.empty(3 * _n_ed, dtype=np.float64)
    _y = np.empty(3 * _n_ed, dtype=np.float64)
    _z = np.empty(3 * _n_ed, dtype=np.float64)
    _x[::3], _y[::3], _z[::3] = _xyz[_i0, 0], _xyz[_i0, 1], _xyz[_i0, 2]
    _x[1::3], _y[1::3], _z[1::3] = _xyz[_i1, 0], _xyz[_i1, 1], _xyz[_i1, 2]
    _x[2::3], _y[2::3], _z[2::3] = np.nan, np.nan, np.nan
else:
    _x = _y = _z = np.array([], dtype=np.float64)

_gx, _gy, _gz = _xyz[_idx, 0], _xyz[_idx, 1], _xyz[_idx, 2]

_fig = go.Figure()
if _n_ed > 0:
    _fig.add_trace(
        go.Scatter3d(
            x=_x,
            y=_y,
            z=_z,
            mode='lines',
            line=dict(color='#2ec4ff', width=2),
            opacity=0.45,
            hoverinfo='skip',
            name=f'edges ({_n_ed:,})',
        )
    )
_fig.add_trace(
    go.Scatter3d(
        x=_gx,
        y=_gy,
        z=_gz,
        mode='markers',
        marker=dict(size=1.8, color='#ff6b9d', opacity=0.6),
        hoverinfo='skip',
        name=f'galaxies ({_n_gal:,})',
    )
)
_fig.update_layout(
    title=(
        f'200 Mpc cube @ ({_c[0]:.0f},{_c[1]:.0f},{_c[2]:.0f}) [{CUBE_MODE}] (comoving Mpc)<br>'
        f'{GRAPH_QC_PREFIX} — edges need both endpoints inside'
    ),
    scene=dict(
        xaxis=dict(title='x [Mpc]', range=[float(_lo[0]), float(_hi[0])], backgroundcolor='#0a0a0c'),
        yaxis=dict(title='y [Mpc]', range=[float(_lo[1]), float(_hi[1])], backgroundcolor='#0a0a0c'),
        zaxis=dict(title='z [Mpc]', range=[float(_lo[2]), float(_hi[2])], backgroundcolor='#0a0a0c'),
        aspectmode='cube',
    ),
    paper_bgcolor='#0a0a0c',
    font=dict(color='white'),
    margin=dict(l=0, r=0, t=60, b=0),
    showlegend=True,
    legend=dict(x=0.02, y=0.98),
    width=900,
    height=750,
)
_fig.show()


In [ ]:
# Box-frame graph edges projected onto the same yz grid as the CWEB overlay
from pathlib import Path as _Pth
import numpy as _np
from matplotlib.collections import LineCollection as _LineCollection
from mpl_toolkits.mplot3d.art3d import Line3DCollection as _Line3DCollection

GRAPH_OVERLAY_DIR = _Pth('/pscratch/sd/d/dkololgi/abacus/graph_constructions')
GRAPH_OVERLAY_PREFIX = 'abacus_mock_alpha_23032026'#'abacus_alpha_boxframe'
MAX_EDGES_DRAW = 150_000  # cap for speed; full slab can have O(1e5)–(1e6) edges
EDGE_LINewidth = 0.35
EDGE_COLOR = '#00d4ff'
EDGE_ALPHA = 0.22
NODE_COLOR = '#ffffff'
NODE_SIZE = 10
RNG_SUB = _np.random.default_rng(123)  # match overlay subsample seed when possible

pts_path = GRAPH_OVERLAY_DIR / f'{GRAPH_OVERLAY_PREFIX}_points_xyz.npy'
edg_path = GRAPH_OVERLAY_DIR / f'{GRAPH_OVERLAY_PREFIX}_edges_combined_idx.npy'
if not pts_path.exists() or not edg_path.exists():
    raise FileNotFoundError(f'Missing {pts_path} or {edg_path}')

pts = _np.load(pts_path).astype(_np.float64)
edges = _np.load(edg_path).astype(_np.int64)
if edges.ndim != 2 or edges.shape[1] != 2:
    raise ValueError(f'Unexpected edges shape {edges.shape}')
if edges.size and int(edges.max()) >= len(pts):
    raise ValueError(f'Edge index out of range: max={int(edges.max())}, n_pts={len(pts)}')

# --- Optional: plot same *edge list* in halo x_com vs RA/Dec/Z->Cartesian (Planck18) ---
# The graph/Delaunay was built on halo `points_xyz.npy`. Sky positions are the same row order as
# the CutSky FITS (Y1|Y5) — use for comparison; edges are NOT the Delaunay of sky-only coords.
GRAPH_PLOT_HALO_XYZ = True
GRAPH_PLOT_SKY_XYZ = False
_pts_sky = None
if GRAPH_PLOT_SKY_XYZ:
    try:
        import fitsio as _fitsio_g
        _cat_g = globals().get('OVERLAY_CATALOG_PATH') or globals().get('CATALOG_OVERRIDE')
        if _cat_g:
            _cpg = _Pth(str(_cat_g)).expanduser().resolve()
            with _fitsio_g.FITS(str(_cpg)) as _fg:
                _cn = _fg[1].get_colnames()
            _um = {n.upper(): n for n in _cn}
            _cols = [_um[k] for k in ('RA', 'DEC', 'IN_Y1', 'IN_Y5') if k in _um]
            _znm = None
            for _zk in ('Z', 'Z_OBS', 'ZOBS'):
                if _zk in _um:
                    _cols.append(_um[_zk])
                    _znm = _um[_zk]
                    break
            if _znm is None:
                raise RuntimeError('No Z column')
            _tb = _fitsio_g.read(str(_cpg), columns=_cols)
            _nm = {n.upper(): n for n in _tb.dtype.names}
            _mk = _np.ones(len(_tb), dtype=bool)
            _i1 = _nm.get('IN_Y1')
            _i5 = _nm.get('IN_Y5')
            if _i1 is not None or _i5 is not None:
                _m = _np.zeros(len(_tb), dtype=bool)
                if _i1 is not None:
                    _m |= _tb[_i1] == 1
                if _i5 is not None:
                    _m |= _tb[_i5] == 1
                if not _m.any():
                    _m[:] = True
                _mk &= _m
            _ra = _tb[_nm['RA']][_mk].astype(_np.float64)
            _de = _tb[_nm['DEC']][_mk].astype(_np.float64)
            _zz = _tb[_znm][_mk].astype(_np.float64)
            if _ra.size != len(pts):
                raise RuntimeError(f'FITS rows {_ra.size:,} != points {len(pts):,}')
            from astropy.coordinates import SkyCoord as _SkyG
            from astropy.cosmology import Planck18 as _cosg
            import astropy.units as _ug
            _comg = _cosg.comoving_distance(_zz).to(_ug.Mpc)
            _sg = _SkyG(ra=_ra * _ug.deg, dec=_de * _ug.deg, distance=_comg, frame='icrs')
            _cg = _sg.cartesian
            _pts_sky = _np.column_stack(
                [_cg.x.to(_ug.Mpc).value, _cg.y.to(_ug.Mpc).value, _cg.z.to(_ug.Mpc).value]
            ).astype(_np.float64)
            print(f'Loaded sky Cartesian (Planck18) for overlay: shape={_pts_sky.shape}')
        else:
            print('GRAPH_PLOT_SKY_XYZ: set OVERLAY_CATALOG_PATH to enable sky Cartesian overlay.')
    except Exception as _eg:
        print('Sky Cartesian overlay skipped:', _eg)
        _pts_sky = None

# Same grid as overlay / T-Web (reload slab meta)
_slabs = discover_slabs(PRIMARY_DIR)
_ngrid = int(_slabs[0].ngrid)
_box = float(_slabs[0].boxsize)
_cell = _box / _ngrid
_ixg = int(_np.clip(round(OVERLAY_X_FRAC * (_ngrid - 1)), 0, _ngrid - 1))
_w = int(OVERLAY_X_HALF_WIDTH_CELLS)

xyz_m = _np.mod(pts, _box)

# --- IllustrisTNG-style thin slab (cf. Fig 5 style): plot two comoving axes in Mpc ---
# Edges: BOTH endpoints inside the slab. Loop: halo x_com vs optional sky Cartesian.
#
# CutSky / boxframe: Gudhi used native halo x_com. `GRAPH_PLOT_SKY_XYZ` overlays the *same* edge
# indices in Planck18 sky Cartesian for comparison (topology fixed; geometry differs).
# Y1/Y5 selection: anisotropic clouds are expected vs periodic cubes.
ILLUSTRIS_SLICE_THICK_MPC = 20.0
ILLUSTRIS_SLICE_CENTER_MPC = None
ILLUSTRIS_SLICE_CENTER_FRAC = None
ILLUSTRIS_SLICE_NORMAL = 'z'
ILLUSTRIS_COORDS_FOR_SLAB = 'native'
MAX_EDGES_ILLUSTRIS = 250_000
_lab_xyz = ('X', 'Y', 'Z')
_axis_map = {'x': 0, 'y': 1, 'z': 2}
_an_ill = _axis_map[str(ILLUSTRIS_SLICE_NORMAL).lower()]
_ap_ill = tuple(i for i in (0, 1, 2) if i != _an_ill)
_half_ill = 0.5 * float(ILLUSTRIS_SLICE_THICK_MPC)

_illustris_runs = []
if GRAPH_PLOT_HALO_XYZ:
    _illustris_runs.append((pts, 'halo x_com (matches Gudhi graph)', 'halo_xcom'))
if GRAPH_PLOT_SKY_XYZ and _pts_sky is not None:
    _illustris_runs.append(
        (_pts_sky, 'RA/Dec/Z -> Cartesian (Planck18); same edges, different embedding', 'sky_cartesian')
    )

for _pts_src, _coord_note, _ill_tag in _illustris_runs:
    _pts_vis = _np.asarray(_pts_src, dtype=_np.float64)
    if str(ILLUSTRIS_COORDS_FOR_SLAB).lower() == 'periodic':
        xyz_vis = _np.mod(_pts_vis, _box)
        _cn = _coord_note + ' | periodic mod L'
    else:
        xyz_vis = _pts_vis
        _cn = _coord_note
    if ILLUSTRIS_SLICE_CENTER_MPC is not None:
        _cen_ill = float(ILLUSTRIS_SLICE_CENTER_MPC)
    elif ILLUSTRIS_SLICE_CENTER_FRAC is not None:
        _cen_ill = float(ILLUSTRIS_SLICE_CENTER_FRAC) * _box
    else:
        _cen_ill = float(_np.nanmedian(xyz_vis[:, _an_ill]))
    _lo_ill, _hi_ill = _cen_ill - _half_ill, _cen_ill + _half_ill
    _coord_ill = xyz_vis[:, _an_ill]
    _fin = _np.isfinite(_coord_ill)
    _in_thin = _fin & (_coord_ill >= _lo_ill) & (_coord_ill <= _hi_ill)
    _ill_e = edges[_in_thin[edges[:, 0]] & _in_thin[edges[:, 1]]]
    print(f'Illustris slab [{_ill_tag}] coords: {_cn}')
    print(
        f'  {_lab_xyz[0]},{_lab_xyz[1]},{_lab_xyz[2]} min/max (finite): '
        f'[{_np.nanmin(xyz_vis[:, 0]):.2f},{_np.nanmax(xyz_vis[:, 0]):.2f}] '
        f'[{_np.nanmin(xyz_vis[:, 1]):.2f},{_np.nanmax(xyz_vis[:, 1]):.2f}] '
        f'[{_np.nanmin(xyz_vis[:, 2]):.2f},{_np.nanmax(xyz_vis[:, 2]):.2f}]'
    )
    print(
        f'  slab ⊥ {ILLUSTRIS_SLICE_NORMAL} ∈ [{_lo_ill:.1f}, {_hi_ill:.1f}] Mpc | '
        f'nodes={int(_in_thin.sum()):,} | edges (both ends in slab)={_ill_e.shape[0]:,}'
    )
    _ill_draw = _ill_e
    if _ill_draw.shape[0] > MAX_EDGES_ILLUSTRIS:
        _ill_draw = _ill_draw[RNG_SUB.choice(_ill_draw.shape[0], size=int(MAX_EDGES_ILLUSTRIS), replace=False)]
        print(f'  subsampled edges for slab figure: {_ill_draw.shape[0]:,}')
    _p0, _p1 = _ap_ill
    _Axy = _np.column_stack([xyz_vis[_ill_draw[:, 0], _p0], xyz_vis[_ill_draw[:, 0], _p1]])
    _Bxy = _np.column_stack([xyz_vis[_ill_draw[:, 1], _p0], xyz_vis[_ill_draw[:, 1], _p1]])
    _segs_xy = _np.stack([_Axy, _Bxy], axis=1).astype(_np.float64)
    _slab_idx_thin = _np.where(_in_thin)[0]
    _nmax_ill = int(OVERLAY_MAX_POINTS) if 'OVERLAY_MAX_POINTS' in globals() else 5000
    if _slab_idx_thin.size > _nmax_ill:
        _nodes_xy = RNG_SUB.choice(_slab_idx_thin, size=_nmax_ill, replace=False)
    else:
        _nodes_xy = _slab_idx_thin
    _fig_ill, _ax_ill = plt.subplots(figsize=(8.8, 8.0), constrained_layout=True)
    _fig_ill.patch.set_facecolor('#000000')
    _ax_ill.set_facecolor('#000000')
    _ax_ill.add_collection(
        _LineCollection(_segs_xy, colors='#7dd3fc', linewidths=0.2, alpha=0.33, zorder=1)
    )
    _ax_ill.scatter(
        xyz_vis[_nodes_xy, _p0],
        xyz_vis[_nodes_xy, _p1],
        s=3.2,
        c='#ff9cda',
        alpha=0.68,
        zorder=2,
        linewidths=0,
    )
    _ax_ill.set_aspect('equal')
    if str(ILLUSTRIS_COORDS_FOR_SLAB).lower() == 'periodic':
        _ax_ill.set_xlim(0.0, _box)
        _ax_ill.set_ylim(0.0, _box)
    else:
        _xs = _np.concatenate([_Axy[:, 0], _Bxy[:, 0], xyz_vis[_nodes_xy, _p0]])
        _ys = _np.concatenate([_Axy[:, 1], _Bxy[:, 1], xyz_vis[_nodes_xy, _p1]])
        _span = max(float(_np.ptp(_xs)), float(_np.ptp(_ys)), 1e-6)
        _pad = 0.03 * _span
        _ax_ill.set_xlim(float(_np.min(_xs)) - _pad, float(_np.max(_xs)) + _pad)
        _ax_ill.set_ylim(float(_np.min(_ys)) - _pad, float(_np.max(_ys)) + _pad)
    _ax_ill.set_xlabel(f'{_lab_xyz[_p0]} [Mpc]', color='0.9')
    _ax_ill.set_ylabel(f'{_lab_xyz[_p1]} [Mpc]', color='0.9')
    _ax_ill.tick_params(colors='0.8')
    for _sp in _ax_ill.spines.values():
        _sp.set_edgecolor('0.35')
    _ax_ill.set_title(
        f'[{_ill_tag}] 2D slab {_lab_xyz[_an_ill]} ∈ [{_lo_ill:.1f}, {_hi_ill:.1f}] Mpc ({ILLUSTRIS_SLICE_THICK_MPC:g} Mpc thick)\n'
        f'{_cn} | {_nodes_xy.size:,} nodes / {len(_segs_xy):,} segs | {GRAPH_OVERLAY_PREFIX}',
        color='0.88',
        fontsize=10,
    )
    _out_ill = OUT_DIR / f'boxframe_graph_illustris_slab_{_ill_tag}_{GRAPH_OVERLAY_PREFIX}.png'
    _fig_ill.savefig(_out_ill, dpi=220, bbox_inches='tight', facecolor=_fig_ill.get_facecolor())
    plt.show()
    print(f'Saved: {_out_ill}')


ix_all = _np.floor(xyz_m[:, 0] / _cell).astype(_np.int32)
iy_all = _np.floor(xyz_m[:, 1] / _cell).astype(_np.int32)
iz_all = _np.floor(xyz_m[:, 2] / _cell).astype(_np.int32)
in_slab = _np.abs(ix_all - _ixg) <= _w

# Edges with both ends in spatial slab (same 3D slab as overlay)
m = in_slab[edges[:, 0]] & in_slab[edges[:, 1]]
slab_e = edges[m]
print(f'Nodes in slab: {int(in_slab.sum()):,} / {len(pts):,}')
print(f'Edges with both ends in slab: {slab_e.shape[0]:,} (before cap)')

_ix0 = ix_all[slab_e[:, 0]]
_ix1 = ix_all[slab_e[:, 1]]
_dix = _np.abs(_ix0.astype(_np.int32) - _ix1.astype(_np.int32))
_iy0, _iy1 = iy_all[slab_e[:, 0]], iy_all[slab_e[:, 1]]
_iz0, _iz1 = iz_all[slab_e[:, 0]], iz_all[slab_e[:, 1]]
_diy = _np.abs(_iy0.astype(_np.int32) - _iy1.astype(_np.int32))
_diz = _np.abs(_iz0.astype(_np.int32) - _iz1.astype(_np.int32))
_len = _np.linalg.norm(_np.mod(pts[slab_e[:, 1]], _box) - _np.mod(pts[slab_e[:, 0]], _box), axis=1)
print(
    'Slab-internal edges: median 3D length (Mpc) = '
    f'{_np.median(_len):.3f} | median grid |Δix|,|Δiy|,|Δiz| = '
    f'{_np.median(_dix):.0f}, {_np.median(_diy):.0f}, {_np.median(_diz):.0f} | '
    f'frac with Δix>0 = {_np.mean(_dix > 0):.3f}'
)
print(
    'NOTE: Default YZ LineCollection THROWS AWAY Δx. Edges that run mostly along x '
    'stack on top of each other in (y,z) — that often looks like \"slabs\" or weird ribbons '
    'even when the 3D Delaunay graph is fine. Use the same-ix panel + 3D view below.'
)

samex = _ix0 == _ix1
slab_e_samex = slab_e[samex]
print(f'Edges with both endpoints in the *same* ix cell: {slab_e_samex.shape[0]:,}')


def _cap_edges(e):
    if e.shape[0] > MAX_EDGES_DRAW:
        return e[RNG_SUB.choice(e.shape[0], size=int(MAX_EDGES_DRAW), replace=False)]
    return e


slab_e_draw = _cap_edges(slab_e)
slab_e_samex_draw = _cap_edges(slab_e_samex)
if slab_e.shape[0] > MAX_EDGES_DRAW:
    print(f'  subsampled ALL slab edges for drawing: {slab_e_draw.shape[0]:,}')
if slab_e_samex.shape[0] > MAX_EDGES_DRAW:
    print(f'  subsampled same-ix edges for drawing: {slab_e_samex_draw.shape[0]:,}')


def _segs_yz(e_arr):
    A = _np.column_stack([iy_all[e_arr[:, 0]], iz_all[e_arr[:, 0]]]).astype(_np.float64)
    B = _np.column_stack([iy_all[e_arr[:, 1]], iz_all[e_arr[:, 1]]]).astype(_np.float64)
    return _np.stack([A, B], axis=1)


segs = _segs_yz(slab_e_draw)
segs_samex = _segs_yz(slab_e_samex_draw) if slab_e_samex_draw.shape[0] else _np.empty((0, 2, 2))

# Node scatter: same slab, same RNG seed as overlay for comparable density
slab_idx = _np.where(in_slab)[0]
if slab_idx.size > int(OVERLAY_MAX_POINTS):
    nodes_plot = RNG_SUB.choice(slab_idx, size=int(OVERLAY_MAX_POINTS), replace=False)
else:
    nodes_plot = slab_idx

fig2, (ax2a, ax2b) = plt.subplots(1, 2, figsize=(13.0, 7.2), constrained_layout=True)
fig2.patch.set_facecolor('#0b0b0b')
for ax in (ax2a, ax2b):
    ax.set_facecolor('#0b0b0b')
    ax.add_collection(
        _LineCollection(
            segs if ax is ax2a else segs_samex,
            colors=EDGE_COLOR,
            linewidths=EDGE_LINewidth,
            alpha=EDGE_ALPHA,
            zorder=1,
        )
    )
    ax.scatter(
        iy_all[nodes_plot],
        iz_all[nodes_plot],
        s=NODE_SIZE,
        c=NODE_COLOR,
        alpha=0.85,
        zorder=2,
        edgecolors='#111111',
        linewidths=0.2,
    )
    ax.set_aspect('equal')
    ax.set_xlabel('grid y', color='0.85')
    ax.set_ylabel('grid z', color='0.85')
    ax.tick_params(colors='0.75')
    for spine in ax.spines.values():
        spine.set_edgecolor('0.4')

ax2a.set_title(
    f'YZ projection — ALL slab-internal edges\n'
    f'{GRAPH_OVERLAY_PREFIX} | ixg={_ixg} ±{_w} | segs={len(segs):,}\n'
    '(Δx dropped → misleading “slab” look)'
)
ax2b.set_title(
    f'YZ — only edges with matching ix on both ends\n'
    f'segs={len(segs_samex):,} (verifies local yz mesh without x-collapse)'
)

if 'OVERLAY_AUTO_ZOOM' in globals() and OVERLAY_AUTO_ZOOM and nodes_plot.size > 0:
    y0, y1 = int(_np.min(iy_all[nodes_plot])), int(_np.max(iy_all[nodes_plot]))
    z0, z1 = int(_np.min(iz_all[nodes_plot])), int(_np.max(iz_all[nodes_plot]))
    mrg = int(max(0, OVERLAY_ZOOM_MARGIN_CELLS)) if 'OVERLAY_ZOOM_MARGIN_CELLS' in globals() else 24
    for ax in (ax2a, ax2b):
        ax.set_xlim(max(0, y0 - mrg), min(_ngrid - 1, y1 + mrg))
        ax.set_ylim(max(0, z0 - mrg), min(_ngrid - 1, z1 + mrg))

out2 = OUT_DIR / f'boxframe_graph_edges_yz_ix{_ixg}_{GRAPH_OVERLAY_PREFIX}.png'
fig2.savefig(out2, dpi=220, bbox_inches='tight', facecolor=fig2.get_facecolor())
plt.show()
print(f'Saved: {out2}')

# --- True 3D segment plot (small subsample of slab edges) ---
MAX_3D_EDGES = 40_000
e3 = slab_e_draw
if e3.shape[0] > MAX_3D_EDGES:
    e3 = e3[RNG_SUB.choice(e3.shape[0], size=int(MAX_3D_EDGES), replace=False)]
_xyz = _np.mod(pts, _box)
_st = _xyz[e3[:, 0]]
_en = _xyz[e3[:, 1]]
segs3 = _np.stack([_st, _en], axis=1)
fig3 = plt.figure(figsize=(7.5, 6.5), constrained_layout=True)
ax3 = fig3.add_subplot(111, projection='3d')
ax3.set_facecolor('#0b0b0b')
fig3.patch.set_facecolor('#0b0b0b')
ax3.add_collection3d(
    _Line3DCollection(
        segs3,
        colors=EDGE_COLOR,
        alpha=0.18,
        linewidths=0.35,
    )
)
ns3 = min(4000, int(nodes_plot.size))
plot_ids = nodes_plot if nodes_plot.size <= ns3 else RNG_SUB.choice(nodes_plot, size=ns3, replace=False)
ax3.scatter(
    _xyz[plot_ids, 0],
    _xyz[plot_ids, 1],
    _xyz[plot_ids, 2],
    c=NODE_COLOR,
    s=8,
    alpha=0.5,
    depthshade=False,
)
ax3.set_xlabel('x [Mpc]', color='0.85')
ax3.set_ylabel('y [Mpc]', color='0.85')
ax3.set_zlabel('z [Mpc]', color='0.85')
ax3.tick_params(colors='0.75')
ax3.set_title(
    f'3D box-frame edges (same slab, Mpc)\n'
    f'≤{len(segs3):,} segments | verify Delaunay adjacency is local in space'
)
ax3.view_init(elev=18, azim=-58)
out3 = OUT_DIR / f'boxframe_graph_edges_3d_ix{_ixg}_{GRAPH_OVERLAY_PREFIX}.png'
fig3.savefig(out3, dpi=200, bbox_inches='tight', facecolor=fig3.get_facecolor())
plt.show()
print(f'Saved: {out3}')

# --- Optional: 3D slab edges in sky Cartesian (same subsample e3 as halo fig above) ---
if _pts_sky is not None and GRAPH_PLOT_SKY_XYZ:
    _xyz_s3 = _np.mod(_pts_sky, _box)
    _st_s = _xyz_s3[e3[:, 0]]
    _en_s = _xyz_s3[e3[:, 1]]
    segs3_s = _np.stack([_st_s, _en_s], axis=1)
    fig3s = plt.figure(figsize=(7.5, 6.5), constrained_layout=True)
    ax3s = fig3s.add_subplot(111, projection='3d')
    ax3s.set_facecolor('#0b0b0b')
    fig3s.patch.set_facecolor('#0b0b0b')
    ax3s.add_collection3d(
        _Line3DCollection(
            segs3_s,
            colors=EDGE_COLOR,
            alpha=0.18,
            linewidths=0.35,
        )
    )
    ax3s.scatter(
        _xyz_s3[plot_ids, 0],
        _xyz_s3[plot_ids, 1],
        _xyz_s3[plot_ids, 2],
        c=NODE_COLOR,
        s=8,
        alpha=0.5,
        depthshade=False,
    )
    ax3s.set_xlabel('x [Mpc]', color='0.85')
    ax3s.set_ylabel('y [Mpc]', color='0.85')
    ax3s.set_zlabel('z [Mpc]', color='0.85')
    ax3s.tick_params(colors='0.75')
    ax3s.set_title(
        f'3D edges in sky Cartesian (Planck18)\n'
        f'≤{len(segs3_s):,} segments | same edge list as halo build',
        color='0.85',
    )
    ax3s.view_init(elev=18, azim=-58)
    out3s = OUT_DIR / f'boxframe_graph_edges_3d_sky_cart_ix{_ixg}_{GRAPH_OVERLAY_PREFIX}.png'
    fig3s.savefig(out3s, dpi=200, bbox_inches='tight', facecolor=fig3s.get_facecolor())
    plt.show()
    print(f'Saved: {out3s}')


### Axis-aligned / “grid” edges vs survey footprint (diagnostics)

**Why you can see two populations of structure:**

1. **Integer `(iy, iz)` / `(ix, …)` plots** — Snapping positions to **grid cells** makes many edges change by **one cell in one direction only**, which draws as **horizontal/vertical** in index space. That is mostly a **visualization artifact**, not necessarily wrong graph math.

2. **`np.mod(pts, box)`** — Folding into `[0, L)` **superposes replicas** of the survey; Delaunay was built in **native** `x_com`, so folded plots can show **misleading** geometry (long chords, lattice-like seams at boundaries).

3. **CutSky geometry** — Y1/Y5 galaxies trace a **survey footprint** in RA/Dec; **dense** regions vs **sparse** halos get very different local Delaunay degree and edge length. **Sparse “outliers”** (or tile seams) can participate in **long** edges that look regular in projection.

4. **Rare real axis alignment in Mpc** — Nearly **exact** shared `x` or `y` in comoving Mpc for many edges would suggest **duplicates**, **rounding**, or **multi-tile** coordinate frames — worth flagging.

The next cell measures **in-plane edge angles in Mpc** (no grid indices), splits **axis-like** vs **diagonal** edges, optionally loads the **same-order FITS** catalog and colors nodes by **kNN distance in RA/Dec** (proxy for “inside footprint” vs sparse/stray).

In [ ]:
# --- Axis-aligned edges + optional RA/Dec footprint (in-plane plots use T-Web voxel grid) ---
# Requires: discover_slabs, PRIMARY_DIR, plt, OUT_DIR from earlier cells.

from pathlib import Path as _PathD
import numpy as _np
import matplotlib.pyplot as _plt
from matplotlib.collections import LineCollection as _LC
from matplotlib.colors import Normalize as _Norm

GRAPH_OVERLAY_DIR_D = _PathD('/pscratch/sd/d/dkololgi/abacus/graph_constructions')
GRAPH_OVERLAY_PREFIX_D = 'abacus_mock_alpha_23032026'

DIAG_SLICE_THICK_MPC = 20.0
DIAG_SLICE_NORMAL = 'z'
DIAG_COORDS = 'native'
DIAG_CENTER_MPC = None
DIAG_AXIS_RATIO_MAX = 0.08
MAX_EDGES_DIAG = 250_000
RNG_D = _np.random.default_rng(123)
# Fixed limits for spatial plots in **grid index** (match overlay / T-Web voxels).
DIAG_GRID_XY_MAX = 2000

# Optional FITS (same Y1|Y5 row order as graph / points_xyz). If None, try OVERLAY_CATALOG_PATH.
CATALOG_FOR_FOOTPRINT = globals().get('OVERLAY_CATALOG_PATH', None)
KNN_K = 8

# --- load graph ---
_pts = _np.load(GRAPH_OVERLAY_DIR_D / f'{GRAPH_OVERLAY_PREFIX_D}_points_xyz.npy').astype(_np.float64)
_edg = _np.load(GRAPH_OVERLAY_DIR_D / f'{GRAPH_OVERLAY_PREFIX_D}_edges_combined_idx.npy').astype(_np.int64)
_slab = discover_slabs(PRIMARY_DIR)
_L = float(_slab[0].boxsize)
_ngrid = int(_slab[0].ngrid)
_cell = _L / _ngrid
if str(DIAG_COORDS).lower() == 'native':
    _xyz = _np.mod(_pts, _L)
else:
    _xyz = _np.asarray(_pts, dtype=_np.float64)

_axis = {'x': 0, 'y': 1, 'z': 2}
_an = _axis[str(DIAG_SLICE_NORMAL).lower()]
_ap = tuple(i for i in (0, 1, 2) if i != _an)
_p0, _p1 = _ap
_half = 0.5 * float(DIAG_SLICE_THICK_MPC)
if DIAG_CENTER_MPC is not None:
    _zc = float(DIAG_CENTER_MPC)
else:
    _zc = float(_np.nanmedian(_xyz[:, _an]))
_lo, _hi = _zc - _half, _zc + _half
_in = _np.isfinite(_xyz).all(axis=1) & (_xyz[:, _an] >= _lo) & (_xyz[:, _an] <= _hi)
em = _in[_edg[:, 0]] & _in[_edg[:, 1]]
_e_sl = _edg[em]
if _e_sl.shape[0] > MAX_EDGES_DIAG:
    _e_sl = _e_sl[RNG_D.choice(_e_sl.shape[0], size=MAX_EDGES_DIAG, replace=False)]

# Axis-like ratio in **Mpc** (unchanged physical diagnostic)
_du = _xyz[_e_sl[:, 1], _p0] - _xyz[_e_sl[:, 0], _p0]
_dv = _xyz[_e_sl[:, 1], _p1] - _xyz[_e_sl[:, 0], _p1]
_len2 = _np.hypot(_du, _dv)
_ratio = _np.minimum(_np.abs(_du), _np.abs(_dv)) / (_len2 + 1e-18)
_axis_like = _ratio < float(DIAG_AXIS_RATIO_MAX)

# Voxel grid indices for plotting (same convention as T-Web overlay)
_xm = _np.mod(_xyz, _L)
_ig = _np.floor(_xm / _cell).astype(_np.int32)
_np.clip(_ig, 0, _ngrid - 1, out=_ig)

print(
    f'Diag slab: ⊥ {DIAG_SLICE_NORMAL} ∈ [{_lo:.1f},{_hi:.1f}] Mpc | coords={DIAG_COORDS} | '
    f'ngrid={_ngrid} | edges={_e_sl.shape[0]:,}'
)
print(
    f'  In-plane: min(|du|,|dv|)/||d|| in [0,1] — small ⇒ axis-like in (X,Y). '
    f'frac axis-like (ratio<{DIAG_AXIS_RATIO_MAX}): {_axis_like.mean():.3f}'
)
print(
    f'  Duplicate exact (x,y,z) in slab: ',
    end='',
)
_slab_idx = _np.where(_in)[0]
if _slab_idx.size:
    _xyz_s = _np.round(_xyz[_slab_idx], 9)
    _uniq = len(_np.unique(_xyz_s, axis=0))
    print(f'{_slab_idx.size:,} nodes, {_uniq:,} unique at 1e-9 Mpc')
else:
    print('no nodes')

# --- segments colored by axis-like (grid coordinates) ---
_st = _ig[_e_sl[:, 0], :][:, (_p0, _p1)].astype(_np.float64)
_en = _ig[_e_sl[:, 1], :][:, (_p0, _p1)].astype(_np.float64)
_segs = _np.stack([_st, _en], axis=1).astype(_np.float64)
_c_aa = _np.where(_axis_like, '#ff5555', '#55aaff')
_fig1, _ax1 = _plt.subplots(1, 1, figsize=(8.2, 7.2), constrained_layout=True)
_fig1.patch.set_facecolor('#050508')
_ax1.set_facecolor('#050508')
for _c, _lab in [('#ff5555', 'axis-like'), ('#55aaff', 'other')]:
    _m = (_c_aa == _c)
    if _m.any():
        _ax1.add_collection(_LC(_segs[_m], colors=_c, linewidths=0.35, alpha=0.35, zorder=1))
_ax1.set_aspect('equal')
_ax1.set_xlim(0.0, float(DIAG_GRID_XY_MAX))
_ax1.set_ylim(0.0, float(DIAG_GRID_XY_MAX))
_labn = ('X', 'Y', 'Z')
_ax1.set_xlabel(f'grid {_labn[_p0]} index', color='0.9')
_ax1.set_ylabel(f'grid {_labn[_p1]} index', color='0.9')
_ax1.set_title(
    f'Edges in slab (voxel grid): red = axis-like in Mpc plane (min(|du|,|dv|)/||d|| < {DIAG_AXIS_RATIO_MAX})\n'
    f'{GRAPH_OVERLAY_PREFIX_D} | axes 0–{int(DIAG_GRID_XY_MAX)} (ngrid={_ngrid})',
    color='0.85',
    fontsize=10,
)
_ax1.tick_params(colors='0.75')
_out_d1 = OUT_DIR / f'boxframe_graph_axislike_edges_{GRAPH_OVERLAY_PREFIX_D}.png'
_fig1.savefig(_out_d1, dpi=200, bbox_inches='tight', facecolor=_fig1.get_facecolor())
_plt.show()
print(f'Saved: {_out_d1}')

# --- histogram of ratio (still Mpc-based) ---
_fig2, _ax2 = _plt.subplots(figsize=(7.5, 4.0), constrained_layout=True)
_ax2.hist(_ratio.clip(0, 1), bins=80, color='#88aacc', edgecolor='none', alpha=0.9)
_ax2.axvline(float(DIAG_AXIS_RATIO_MAX), color='r', ls='--', lw=1, label='axis-like cut')
_ax2.set_xlabel('min(|Δu|,|Δv|) / ||Δu,Δv||  (0 = horizontal/vertical in plane)', fontsize=10)
_ax2.set_ylabel('edge count')
_ax2.set_title('In-plane edge “axis-ness” (continuous Mpc); spikes at 0 ⇒ H/V structure')
_ax2.legend()
_out_d2 = OUT_DIR / f'boxframe_graph_edge_axis_ratio_hist_{GRAPH_OVERLAY_PREFIX_D}.png'
_fig2.savefig(_out_d2, dpi=160, bbox_inches='tight')
_plt.show()
print(f'Saved: {_out_d2}')

# --- optional FITS: kNN distance in RA/Dec for footprint vs stray ---
_foot_scatter = None
if CATALOG_FOR_FOOTPRINT is not None and _PathD(str(CATALOG_FOR_FOOTPRINT)).exists():
    try:
        import fitsio as _fitsio
        from scipy.spatial import cKDTree as _cKDTree

        with _fitsio.FITS(str(CATALOG_FOR_FOOTPRINT)) as _ff:
            _cavail = _ff[1].get_colnames()
        _um = {n.upper(): n for n in _cavail}
        _cols = [_um[k] for k in ('RA', 'DEC', 'IN_Y1', 'IN_Y5') if k in _um]
        if len(_cols) < 2:
            raise RuntimeError('FITS missing RA/DEC')
        _t = _fitsio.read(str(CATALOG_FOR_FOOTPRINT), columns=_cols)
        _names = {n.upper(): n for n in _t.dtype.names}
        _iy1 = _names.get('IN_Y1')
        _iy5 = _names.get('IN_Y5')
        _mask = _np.zeros(len(_t), dtype=bool)
        if _iy1 is not None:
            _mask |= _t[_iy1] == 1
        if _iy5 is not None:
            _mask |= _t[_iy5] == 1
        if not _mask.any():
            _mask[:] = True
        _ra = _t[_names['RA']][_mask].astype(_np.float64)
        _dec = _t[_names['DEC']][_mask].astype(_np.float64)
        if len(_ra) != len(_pts):
            print(
                f'WARNING: FITS Y1|Y5 rows ({len(_ra):,}) != points ({len(_pts):,}); '
                'skip footprint kNN (check alignment order).'
            )
        else:
            _xy_s = _np.column_stack([_ra, _dec])
            _nk = min(KNN_K + 1, len(_xy_s))
            _nref = min(400_000, max(50_000, len(_xy_s) // 50))
            _ref_i = RNG_D.choice(len(_xy_s), size=_nref, replace=False)
            _tree = _cKDTree(_xy_s[_ref_i])
            _q_rd = _xy_s[_slab_idx]
            _k_q = min(KNN_K, _nref)
            _d_nn, _ = _tree.query(_q_rd, k=_k_q)
            _d_knn_slab = _d_nn[:, -1]
            _foot_scatter = (_d_knn_slab, _in, _slab_idx)
            _fig3, _ax3 = _plt.subplots(1, 1, figsize=(8.2, 7.0), constrained_layout=True)
            _sc = _ax3.scatter(
                _ig[_slab_idx, _p0].astype(_np.float64),
                _ig[_slab_idx, _p1].astype(_np.float64),
                c=_d_knn_slab,
                s=4,
                cmap='magma',
                alpha=0.75,
                norm=_Norm(
                    vmin=_np.percentile(_d_knn_slab, 2),
                    vmax=_np.percentile(_d_knn_slab, 98),
                ),
            )
            _plt.colorbar(_sc, ax=_ax3, label=f'{_k_q}-NN dist to ref sample in RA/Dec (deg)\n(large ⇒ sparse / stray)')
            _ax3.set_aspect('equal')
            _ax3.set_xlim(0.0, float(DIAG_GRID_XY_MAX))
            _ax3.set_ylim(0.0, float(DIAG_GRID_XY_MAX))
            _ax3.set_xlabel(f'grid {_labn[_p0]} index', color='0.9')
            _ax3.set_ylabel(f'grid {_labn[_p1]} index', color='0.9')
            _ax3.set_facecolor('#0a0a0c')
            _ax3.set_title(
                'Nodes in slab colored by kNN distance (RA,Dec)\n'
                f'Axes 0–{int(DIAG_GRID_XY_MAX)} voxel indices | ngrid={_ngrid}',
                color='0.9',
            )
            _out_d3 = OUT_DIR / f'boxframe_graph_footprint_knn_{GRAPH_OVERLAY_PREFIX_D}.png'
            _fig3.savefig(_out_d3, dpi=200, bbox_inches='tight', facecolor=_fig3.get_facecolor())
            _plt.show()
            print(f'Saved: {_out_d3}')
    except Exception as _ex:
        print('Footprint / kNN block skipped:', _ex)
else:
    print('Set CATALOG_FOR_FOOTPRINT or OVERLAY_CATALOG_PATH to enable RA/Dec kNN footprint plot.')


### Mock galaxies in sky coordinates (raw observables)

Scatter of **RA / Dec** from the CutSky FITS (optionally the same **Y1|Y5** selection as the graph). No comoving conversion — only what enters the mock as angular positions. Subsample for speed; increase `SKY_PLOT_MAX_POINTS` if needed.

In [ ]:
# --- Mock galaxies: RA / Dec only (raw sky observables) ---
import numpy as _np
import matplotlib.pyplot as _plt
from pathlib import Path as _PathS

SKY_PLOT_MAX_POINTS = 200_000
SKY_USE_Y1Y5_ONLY = True
SKY_SEED = 42
SKY_COLOR_BY = 'Z'  # 'Z' (observed redshift) | 'none' (single color)
RNG_SKY = _np.random.default_rng(SKY_SEED)

_cat_path = globals().get('OVERLAY_CATALOG_PATH') or globals().get('CATALOG_OVERRIDE')
if not _cat_path:
    raise ValueError('Set OVERLAY_CATALOG_PATH (overlay cell) or CATALOG_OVERRIDE (config).')
_cat_path = _PathS(str(_cat_path)).expanduser().resolve()
if not _cat_path.exists():
    raise FileNotFoundError(_cat_path)

import fitsio as _fitsio_s


def _read_cutsky_sky_cols(path):
    # Read only RA/Dec/Y1/Y5/Z — full FITS (all columns) can OOM the kernel.
    with _fitsio_s.FITS(str(path)) as _f:
        _names = _f[1].get_colnames()
    _umap = {n.upper(): n for n in _names}
    _cols = []
    for _k in ('RA', 'DEC', 'IN_Y1', 'IN_Y5'):
        if _k in _umap:
            _cols.append(_umap[_k])
    if len(_cols) < 2:
        raise KeyError(f'Need at least RA,DEC in FITS; sample: {_names[:25]}')
    for _zk in ('Z', 'Z_OBS', 'ZOBS'):
        if _zk in _umap:
            _cols.append(_umap[_zk])
            break
    return _fitsio_s.read(str(path), columns=_cols)


_tbl = _read_cutsky_sky_cols(_cat_path)
_names = {n.upper(): n for n in _tbl.dtype.names}
_ra_n = _names.get('RA')
_dec_n = _names.get('DEC')
if _ra_n is None or _dec_n is None:
    raise KeyError(f'Need RA and DEC columns; have: {list(_tbl.dtype.names)}')

_mask = _np.ones(len(_tbl), dtype=bool)
if SKY_USE_Y1Y5_ONLY:
    _iy1 = _names.get('IN_Y1')
    _iy5 = _names.get('IN_Y5')
    _mask &= _np.zeros(len(_tbl), dtype=bool)
    if _iy1 is not None:
        _mask |= _tbl[_iy1] == 1
    if _iy5 is not None:
        _mask |= _tbl[_iy5] == 1
    if not _mask.any():
        _mask[:] = True

_ra = _tbl[_ra_n][_mask].astype(_np.float64)
_dec = _tbl[_dec_n][_mask].astype(_np.float64)
_n = _ra.size
if _n > SKY_PLOT_MAX_POINTS:
    _pick = RNG_SKY.choice(_n, size=int(SKY_PLOT_MAX_POINTS), replace=False)
    _ra = _ra[_pick]
    _dec = _dec[_pick]
    _n_plot = _ra.size
else:
    _n_plot = _n

_c = '#66ccff'
if str(SKY_COLOR_BY).lower() == 'z':
    _z_n = _names.get('Z') or _names.get('Z_OBS') or _names.get('ZOBS')
    if _z_n is not None:
        _z_all = _tbl[_z_n][_mask].astype(_np.float64)
        if _n > SKY_PLOT_MAX_POINTS:
            _z_all = _z_all[_pick]
        _c = _z_all

_fig, _ax = _plt.subplots(figsize=(10.0, 5.2), constrained_layout=True)
_fig.patch.set_facecolor('#0a0a0c')
_ax.set_facecolor('#0a0a0c')
if _np.isscalar(_c) or isinstance(_c, str):
    _ax.scatter(_ra, _dec, s=0.35, c=_c, alpha=0.65, linewidths=0, rasterized=True)
else:
    _sc = _ax.scatter(_ra, _dec, s=0.35, c=_c, cmap='plasma', alpha=0.7, linewidths=0, rasterized=True)
    _cb = _fig.colorbar(_sc, ax=_ax, shrink=0.85, pad=0.02)
    _cb.set_label('Z (observed)', color='0.85')
    _cb.ax.tick_params(colors='0.75')

_ax.set_xlabel('RA [deg]', color='0.9', fontsize=11)
_ax.set_ylabel('Dec [deg]', color='0.9', fontsize=11)
_ax.tick_params(colors='0.75')
_sky_title = (
    'Mock galaxies (sky only) | '
    + str(_cat_path.name)
    + f' | plotted {_n_plot:,} / {_n:,}'
    + (' (Y1|Y5)' if SKY_USE_Y1Y5_ONLY else ' (all rows)')
    + (f' | color={SKY_COLOR_BY}' if not (_np.isscalar(_c) or isinstance(_c, str)) else '')
)
_ax.set_title(_sky_title, color='0.88', fontsize=10)
for _sp in _ax.spines.values():
    _sp.set_edgecolor('0.35')

_out_sky = OUT_DIR / f'mock_galaxies_sky_ra_dec_{_cat_path.stem[:48]}.png'
_fig.savefig(_out_sky, dpi=200, bbox_inches='tight', facecolor=_fig.get_facecolor())
_plt.show()
print(f'Saved: {_out_sky}')


### Mock galaxies: Cartesian from sky vs from halo linkage

Uses the **same random subsample** as the RA/Dec scatter cell (`SKY_PLOT_MAX_POINTS`, `SKY_USE_Y1Y5_ONLY`, `SKY_SEED`).

1. **Sky → Cartesian:** `RA`, `Dec`, observed `Z` → comoving distance with **Planck18** → Cartesian Mpc (same construction as `build_abacus_graph.py` catalog mode).
2. **Halo → Cartesian:** `FILE_NUM` + `HALO_INDEX` → host halo `x_com` from `halo_info_XXX.asdf` (same as `export_cutsky_boxframe_points.py` / graph `points_xyz.npy`).

Override halo directory with env **`TNG_ABACUS_HALO_INFO_DIR`** if needed.

In [ ]:
# --- Same subsample: Cartesian from (RA, Dec, Z_obs) vs from (FILE_NUM, HALO_INDEX) -> x_com ---
import os
import numpy as _np
import matplotlib.pyplot as _plt
from pathlib import Path as _PathC

SKY_PLOT_MAX_POINTS = int(globals().get('SKY_PLOT_MAX_POINTS', 200_000))
SKY_USE_Y1Y5_ONLY = bool(globals().get('SKY_USE_Y1Y5_ONLY', True))
SKY_SEED = int(globals().get('SKY_SEED', 42))
HALO_POS_FIELD = 'x_com'
RNG_C = _np.random.default_rng(SKY_SEED)

_cat_path = globals().get('OVERLAY_CATALOG_PATH') or globals().get('CATALOG_OVERRIDE')
if not _cat_path:
    raise ValueError('Set OVERLAY_CATALOG_PATH or CATALOG_OVERRIDE')
_cat_path = _PathC(str(_cat_path)).expanduser().resolve()
if not _cat_path.exists():
    raise FileNotFoundError(_cat_path)

_ab_base = os.environ.get(
    'TNG_ABACUS_BASE',
    '/global/cfs/cdirs/desi/public/cosmosim/AbacusSummit/AbacusSummit_base_c000_ph000',
)
HALO_INFO_DIR = _PathC(
    os.environ.get('TNG_ABACUS_HALO_INFO_DIR', str(_PathC(_ab_base) / 'halos' / 'z0.200' / 'halo_info'))
)

import fitsio as _fitsio_c


def _read_cutsky_cart_cols(path):
    # Minimal FITS columns for sky+halo Cartesian — avoids loading full catalog (OOM).
    with _fitsio_c.FITS(str(path)) as _f:
        _names = _f[1].get_colnames()
    _umap = {n.upper(): n for n in _names}
    _need = ('RA', 'DEC', 'FILE_NUM', 'HALO_INDEX', 'IN_Y1', 'IN_Y5')
    _cols = [_umap[k] for k in _need if k in _umap]
    _z_key = None
    for _zk in ('Z', 'Z_OBS', 'ZOBS'):
        if _zk in _umap:
            _cols.append(_umap[_zk])
            _z_key = _zk
            break
    if len(_cols) < 4:
        raise KeyError(f'Need RA,DEC,FILE_NUM,HALO_INDEX (+Z); sample: {_names[:30]}')
    if _z_key is None:
        raise KeyError('Need Z or Z_OBS in FITS')
    return _fitsio_c.read(str(path), columns=_cols)


_tbl = _read_cutsky_cart_cols(_cat_path)
_names = {n.upper(): n for n in _tbl.dtype.names}
_ra_n = _names['RA']
_dec_n = _names['DEC']
_z_n = _names.get('Z') or _names.get('Z_OBS') or _names.get('ZOBS')
_fn_n = _names['FILE_NUM']
_hi_n = _names['HALO_INDEX']
if _z_n is None:
    raise KeyError('Need Z (or Z_OBS) in FITS')


_mask = _np.ones(len(_tbl), dtype=bool)
if SKY_USE_Y1Y5_ONLY:
    _m = _np.zeros(len(_tbl), dtype=bool)
    _iy1 = _names.get('IN_Y1')
    _iy5 = _names.get('IN_Y5')
    if _iy1 is not None:
        _m |= _tbl[_iy1] == 1
    if _iy5 is not None:
        _m |= _tbl[_iy5] == 1
    if not _m.any():
        _m[:] = True
    _mask &= _m

_idx = _np.flatnonzero(_mask)
if _idx.size > SKY_PLOT_MAX_POINTS:
    _pick = RNG_C.choice(_idx.size, size=int(SKY_PLOT_MAX_POINTS), replace=False)
    _sub = _idx[_pick]
else:
    _sub = _idx

_ra = _tbl[_ra_n][_sub].astype(_np.float64)
_dec = _tbl[_dec_n][_sub].astype(_np.float64)
_z_obs = _tbl[_z_n][_sub].astype(_np.float64)
_file_num = _tbl[_fn_n][_sub].astype(_np.int32)
_halo_ix = _tbl[_hi_n][_sub].astype(_np.int64)

# (1) RA/Dec/Z_obs -> Cartesian Mpc (match build_abacus_graph catalog path)
from astropy.coordinates import SkyCoord as _SkyCoord
from astropy.cosmology import Planck18 as _cosmo
import astropy.units as _u

_com = _cosmo.comoving_distance(_z_obs).to(_u.Mpc)
_sc = _SkyCoord(
    ra=_ra * _u.deg,
    dec=_dec * _u.deg,
    distance=_com,
    frame='icrs',
)
_cart = _sc.cartesian
_xyz_sky = _np.column_stack(
    [
        _cart.x.to(_u.Mpc).value,
        _cart.y.to(_u.Mpc).value,
        _cart.z.to(_u.Mpc).value,
    ]
).astype(_np.float64)

# (2) FILE_NUM + HALO_INDEX -> x_com from CompaSO halo_info
_xyz_halo = _np.full_like(_xyz_sky, _np.nan)
try:
    from abacusnbody.data.compaso_halo_catalog import CompaSOHaloCatalog as _CompaSOHaloCatalog
except ImportError as _e:
    raise RuntimeError('Install abacusnbody (abacusutils) for halo x_com.') from _e

for _fn in _np.unique(_file_num):
    _sel = _file_num == _fn
    _hp = HALO_INFO_DIR / f'halo_info_{int(_fn):03d}.asdf'
    if not _hp.exists():
        print(f'WARNING: missing {_hp}')
        continue
    try:
        _cat = _CompaSOHaloCatalog(
            str(_hp),
            fields=[HALO_POS_FIELD],
            subsamples=False,
            convert_units=True,
            verbose=False,
        )
    except (TypeError, ValueError):
        try:
            _cat = _CompaSOHaloCatalog(
                str(_hp),
                fields=[HALO_POS_FIELD],
                cleaned=True,
                convert_units=True,
                verbose=False,
            )
        except (TypeError, ValueError):
            _cat = _CompaSOHaloCatalog(str(_hp), cleaned=True)
    _arr = _np.asarray(_cat.halos[HALO_POS_FIELD], dtype=_np.float64)
    _nh = _arr.shape[0]
    _hidx = _halo_ix[_sel]
    _rows = _np.where(_sel)[0]
    _ok = (_hidx >= 0) & (_hidx < _nh)
    if _np.any(_ok):
        _xyz_halo[_rows[_ok]] = _arr[_hidx[_ok]]

_n_bad = int(_np.sum(~_np.isfinite(_xyz_halo).all(axis=1)))
print(
    f'Subsample: {_sub.size:,} rows | halo x_com NaN: {_n_bad:,} | '
    f'halo_info_dir={HALO_INFO_DIR}'
)


def _plot_xy_xz(xyz, title, c, clabel, fname):
    _fig, _axes = _plt.subplots(1, 2, figsize=(12.0, 5.2), constrained_layout=True)
    _fig.patch.set_facecolor('#0a0a0c')
    for _a in _axes:
        _a.set_facecolor('#0a0a0c')
    if _np.ndim(c) == 0 or isinstance(c, str):
        _axes[0].scatter(xyz[:, 0], xyz[:, 1], s=0.35, c=c, alpha=0.65, linewidths=0, rasterized=True)
        _axes[1].scatter(xyz[:, 0], xyz[:, 2], s=0.35, c=c, alpha=0.65, linewidths=0, rasterized=True)
    else:
        _sc0 = _axes[0].scatter(
            xyz[:, 0], xyz[:, 1], s=0.35, c=c, cmap='plasma', alpha=0.7, linewidths=0, rasterized=True
        )
        _axes[1].scatter(
            xyz[:, 0], xyz[:, 2], s=0.35, c=c, cmap='plasma', alpha=0.7, linewidths=0, rasterized=True
        )
        _cb = _fig.colorbar(_sc0, ax=_axes, shrink=0.85, pad=0.02)
        _cb.set_label(clabel, color='0.85')
        _cb.ax.tick_params(colors='0.75')
    _axes[0].set_aspect('equal')
    _axes[1].set_aspect('equal')
    _axes[0].set_xlabel('X [Mpc]', color='0.9')
    _axes[0].set_ylabel('Y [Mpc]', color='0.9')
    _axes[1].set_xlabel('X [Mpc]', color='0.9')
    _axes[1].set_ylabel('Z [Mpc]', color='0.9')
    for _a in _axes:
        _a.tick_params(colors='0.75')
        for _sp in _a.spines.values():
            _sp.set_edgecolor('0.35')
    _fig.suptitle(title, color='0.88', fontsize=11)
    _out = OUT_DIR / fname
    _fig.savefig(_out, dpi=200, bbox_inches='tight', facecolor=_fig.get_facecolor())
    _plt.show()
    print(f'Saved: {_out}')


# Single-line titles (avoid notebook JSON newline/backslash issues)
_title_sky = (
    'Cartesian from RA, Dec, Z_obs (Planck18 comoving) | '
    + f'{_cat_path.name} | n={_sub.size:,}'
)
_plot_xy_xz(
    _xyz_sky,
    _title_sky,
    _z_obs,
    'Z (observed)',
    f'mock_cartesian_from_sky_{_cat_path.stem[:40]}.png',
)

_title_halo = (
    f'Cartesian from FILE_NUM + HALO_INDEX -> {HALO_POS_FIELD} | '
    + f'{_cat_path.name} | n={_sub.size:,} | NaN={_n_bad:,}'
)
_plot_xy_xz(
    _xyz_halo,
    _title_halo,
    _z_obs,
    'Z (observed)',
    f'mock_cartesian_from_halo_{HALO_POS_FIELD}_{_cat_path.stem[:40]}.png',
)


In [ ]:
import json, numpy as np, pandas as pd
from pathlib import Path

GRAPH = Path("/pscratch/sd/d/dkololgi/abacus/graph_constructions")
prefix = "abacus_alpha_boxframe"

meta = json.loads((GRAPH / f"{prefix}_metadata.json").read_text())
pts = np.load(GRAPH / f"{prefix}_points.npy")[:, :3]  # or points_xyz
edges = np.load(GRAPH / f"{prefix}_edges_combined_idx.npy")
deg = np.bincount(edges.ravel(), minlength=len(pts))

print("n_pts", len(pts), "n_edges", len(edges), "mean deg", 2*len(edges)/len(pts))
print("deg min/max", deg.min(), deg.max(), "median", np.median(deg))

# random edge length sample
rng = np.random.default_rng(0)
e = edges[rng.choice(len(edges), size=50_000, replace=False)]
d = np.linalg.norm(pts[e[:,0]] - pts[e[:,1]], axis=1)
print("edge length stats (Mpc): min med max", d.min(), np.median(d), d.max())

nf = pd.read_parquet(GRAPH / f"{prefix}_cugraph_node_features.parquet", columns=["Degree"])
print("parquet Degree vs recomputed (corr):", np.corrcoef(nf["Degree"].values, deg)[0,1])